# Phase 5 — Seven-well TWT correlation with interactive QC navigation
Cell 11 supports interactive TWT window zoom/panning, unfilled gamma-ray tracks, and larger marker labels. All previous crossover screening and LAS input logic are retained.


Cell 12 exports a wide X/Y/TWT marker matrix after strict deviation survey QC. Missing survey coordinates stop export rather than inventing positions.

**Updated:** Cells 3–4 now require deviation/vertical MD→TVDSS and checkshot TVDSS→TWT conversion before any correlation. Cell 12 reuses the same trajectory model for marker X/Y. Missing absolute XY remains blank rather than guessed.

**Stratigraphic control:** Cells 9, 10 and 12 use the interpreter-specified shallow-to-deep marker sequence configured in Cell 1. Markers not in that sequence are flagged for review and appended after ranked markers in CSV outputs. Original measured picks are never shifted automatically.

In [1]:
# CELL 1 — Phase 5 TWT well correlation
from pathlib import Path
import re, warnings, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import PchipInterpolator
from scipy.optimize import linear_sum_assignment
from IPython.display import display
import ipywidgets as widgets

ROOT=Path('/home/apgce_team_08/link-to-apgce_team_08')
PHASE4_ROOT=ROOT/'Well_To_Predict_Phase4_Output'
PHASE3_ROOT=ROOT/'Well_4_6_7_Phase3_FluidReservoir_Output'
PHASE2_ROOT=ROOT/'Well_4_6_7_Phase2_From_PhysicsLossNN_Output'
OUTPUT_ROOT=ROOT/'Phase5_Well_Correlation_Output'
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
# Well 3 is outside seismic coverage and excluded from all processing.
WELL_NAMES=[f'Well_{i}' for i in (1,2,4,5,6,7,8)]
KB_M={'Well_1':9.45,'Well_2':22.,'Well_4':21.978,
      'Well_5':22.,'Well_6':22.,'Well_7':29.,'Well_8':29.8}
MARKER_MATRIX_PATH=ROOT/'Phase5_Step5_K20_Merged_Marker_Matrix.csv'
# Upload the supplied Phase 5 Step 5 CSV to MARKER_MATRIX_PATH before running.
# Explicit overrides for ambiguous training-well CSV locations:
# EXPLICIT ORIGINAL LAS + PHASE 1 COMPOSITE INPUTS (seven wells only)
WELL_INPUT_PATHS={
 'Well_1':{'las':ROOT/'Well_To_Predict/Well-1.las','composite':PHASE4_ROOT/'Well_1/Well_1_Phase1_Predicted_Composites.csv'},
 'Well_2':{'las':ROOT/'Well_To_Predict/Well-2.las','composite':PHASE4_ROOT/'Well_2/Well_2_Phase1_Predicted_Composites.csv'},
 'Well_4':{'las':ROOT/'Well_4/Well-4.las','composite':PHASE2_ROOT/'Well_4_Phase1_NN_Predicted_Composite.csv'},
 'Well_5':{'las':ROOT/'Well_To_Predict/Well-5.las','composite':PHASE4_ROOT/'Well_5/Well_5_Phase1_Predicted_Composites.csv'},
 'Well_6':{'las':ROOT/'Well_6/Well-6.las','composite':PHASE2_ROOT/'Well_6_Phase1_NN_Predicted_Composite.csv'},
 'Well_7':{'las':ROOT/'Well_7/Well-7.las','composite':PHASE2_ROOT/'Well_7_Phase1_NN_Predicted_Composite.csv'},
 'Well_8':{'las':ROOT/'Well_To_Predict/Well-8.las','composite':PHASE4_ROOT/'Well_8/Well_8_Phase1_Predicted_Composites.csv'},
}
# Observed LAS curves are primary; predicted composite curves are supplementary.
# LAS depth is measured depth. Join via composite MDKB_m -> TVDSS_m; no blind MD=TVDSS assumption.
GR_PRIMARY_REQUIRED=True
MIN_LAS_GR_COVERAGE=0.40
WELL_RESULT_OVERRIDES={} # {'Well_4':Path('/.../Well_4_Results.csv')}
WELL_MARKER_OVERRIDES={
    'Well_4':ROOT/'Well_4'/'Well-4_Markers.txt',
    'Well_6':ROOT/'Well_6'/'Well-6_Markers.txt',
    'Well_7':ROOT/'Well_7'/'Well-7_Markers.txt',
}
NEW_WELL_MARKER_OVERRIDES={} # optional, if auto-discovery fails
EXCLUDE_MARKERS={'SEABED','MTSB_K_SHALE'}
REGIONAL_COAL_ANCHOR='I-50_COAL'
MIN_COAL_THICKNESS_M=1.5  # EDITABLE: OTHER coal beds must be thicker than this (metres)
# I-50_COAL is an established regional marker and is EXEMPT from this filter.
COAL_PROBABILITY_THRESHOLD=0.65
COAL_GAP_BRIDGE_M=0.7
MAX_COAL_MATCH_RESIDUAL_M=35.0
MAX_MARKER_ADJUST_M=12.0
MAX_MARKER_PROPOSE_DISTANCE_M=35.0
MAX_WARP_LOCAL_SLOPE=1.35
MIN_WARP_LOCAL_SLOPE=0.70
MAX_WARP_EXTRAPOLATION_M=120.0
MIN_CORRELATION_SCORE=0.55
MAX_UNCERTAINTY_M=25.0
COAL_GR_CUTOFF=65.0  # fallback when no lithology labels; verify local log response
COAL_RHOB_CUTOFF=1.85 # g/cc fallback; only applied with GR
REFERENCE_WELL='Well_4'
SHOW_QC=True
print('Phase 5 output:',OUTPUT_ROOT)
# TWT-domain tolerances (ms); independently calibrated from former depth tolerances
MAX_COAL_MATCH_RESIDUAL_MS=30.0
MAX_MARKER_ADJUST_MS=10.0
MAX_MARKER_PROPOSE_DISTANCE_MS=30.0
MAX_WARP_EXTRAPOLATION_MS=100.0

# Weighted multilog similarity (weights normalized over curves actually available)
LOG_CORRELATION_WEIGHTS={'GR_LAS':0.65,'RHOB_LAS':0.15,'DTC_LAS':0.10,
                         'FCOAL':0.05,'VCLB':0.03,'DENS_P':0.02}
GR_PRIMARY_REQUIRED=True # Require observed LAS gamma ray for multilog confidence
# Minimum separation between consecutive regional markers in local TWT (ms).
MIN_MARKER_SEPARATION_MS=2.0  # EDITABLE; does not alter measured marker TWT
PANEL_FONT_SIZE=12
PANEL_MARKER_FONT_SIZE=9
PANEL_DPI=220
TRAJECTORY_ROOT=Path('/home/apgce_team_08/To_Overlay_Horizon')
DEVIATION_FILE_OVERRIDES={}
CHECKSHOT_OVERRIDES={}
WELLHEAD_XY_OVERRIDES={} # {'Well_1': (X_easting_m, Y_northing_m)}
# No deviation file => vertical. Missing wellhead XY => blank X/Y, not invented.
MAX_COMPOSITE_TVDSS_MISMATCH_M=5.0
MIN_VALID_TWT_COVERAGE=0.40
print("Cell 1 completed")

# Interpreter-approved stratigraphic order: shallowest to deepest.
# This order overrides reference-well TWT-derived ranking for crossover QC.
STRATIGRAPHIC_ORDER = [
    'BASAL_B_UNCONFORMITY',
    'TOP_GROUP_H',
    'TOP_GROUP_I',
    'I-15.0L',
    'I-25',
    'I-27.0',
    'I-35',
    'I-35_LOWER',
    'I-40',
    'I-45',
    'I-50_COAL',
    'I-68',
    'I-70',
    'I-75',
    'I-85',
    'I-95',
    'I-100',
    'I-110_COAL',
    'I-115',
    'I-118',
    'I-120',
    'TOP_GROUP_J',
    'J-3',
    'J-10',
    'J-15',
    'J-20',
    'J-25',
    'TOP_GROUP_K',
    'MTSB_K_SHALE',
    'TOP_GROUP_K_SANDS',
    'K-20',
    'K-22',
    'K-25_UPPER',
    'K-25_LOWER',
    'K-28',
    'K-30',
    'TOP_GROUP_L',
    'L-10',
    'L-30',
    'L-35',
    'L-40',
    'L-60',
    'L-70',
    'TOP_M_SHALE',
    'M-1',
    'M-2',
]
STRATIGRAPHIC_RANK = {name: i for i, name in enumerate(STRATIGRAPHIC_ORDER)}
assert len(STRATIGRAPHIC_RANK) == len(STRATIGRAPHIC_ORDER), 'Duplicate stratigraphic names'


Phase 5 output: /home/apgce_team_08/link-to-apgce_team_08/Phase5_Well_Correlation_Output
Cell 1 completed


In [2]:
# CELL 2 — Phase 5 TWT well correlation
# Load the approved Step 5 marker matrix and build alias lookup.
if not MARKER_MATRIX_PATH.is_file():
    raise FileNotFoundError(f'Place the Step 5 approved marker CSV at: {MARKER_MATRIX_PATH}')
MATRIX=pd.read_csv(MARKER_MATRIX_PATH,dtype=str).fillna('')
required={'Marker_Name',*WELL_NAMES}
if not required.issubset(MATRIX.columns):
    raise ValueError(f'Marker matrix missing columns: {required-set(MATRIX.columns)}')
MATRIX['Marker_Name']=MATRIX['Marker_Name'].str.strip().str.upper()
MATRIX=MATRIX[~MATRIX.Marker_Name.isin(EXCLUDE_MARKERS)].copy()
if MATRIX.Marker_Name.duplicated().any():
    raise ValueError('Duplicate canonical marker names in Step 5 matrix')
ALIASES={}
for _,row in MATRIX.iterrows():
    for well in WELL_NAMES:
        raw=row[well].strip().upper()
        if not raw:continue
        for alias in [x.strip() for x in raw.split(';') if x.strip()]:
            key=(well,alias)
            if key in ALIASES and ALIASES[key]!=row.Marker_Name:
                raise ValueError(f'Conflicting aliases: {key}')
            ALIASES[key]=row.Marker_Name
# Well_3 may exist in the input CSV but is deliberately ignored throughout.
display(MATRIX.head(8))
print('Approved canonical markers:',len(MATRIX),'aliases:',len(ALIASES))

print("Cell 2 completed")


,Marker_Name,Well_1,Well_2,Well_3,Well_4,Well_5,Well_6,Well_7,Well_8,Wells_Present,Status
0,BASAL_B_UNCONFORMITY,,BASAL_B_UNCONFORMITY,,,,,,,1,SINGLE_WELL_OR_VARIANT
1,I-100,,,,I-100,I-100,I-100,I-100,I-100,5,EXACT_MERGED
2,I-110_COAL,,I-110_COAL,,,,,,,1,SINGLE_WELL_OR_VARIANT
3,I-115,,,,I-115,I-115,I-115,,,3,EXACT_MERGED
4,I-118,,,,,I-118,I-118,,,2,EXACT_MERGED
5,I-120,,,,I-120,I-120,I-120,,,3,EXACT_MERGED
6,I-15.0L,,,,,,,,I-15.0L,1,SINGLE_WELL_OR_VARIANT
7,I-25,,I-25_TOP,,I-25,I-25,I-25,,,4,STEP4_ALIAS_MERGED


Approved canonical markers: 45 aliases: 96
Cell 2 completed


In [3]:
# CELL 3 — Phase 5 TWT well correlation
# File discovery is intentionally transparent: never silently use another well's CSV.
RESULT_COLUMNS=('TVDSS_m','TWT_ms','MDKB_m')
def pick_result_file(well):
    """Only the user-specified Phase 1 composite for each well is permitted."""
    if well not in WELL_INPUT_PATHS:raise ValueError(f'Excluded or unknown well: {well}')
    path=WELL_INPUT_PATHS[well]['composite']
    if not path.is_file():raise FileNotFoundError(f'{well}: composite not found: {path}')
    return path

# Load original measured LAS curves with explicit source/units.
LAS_GR_NAMES=('GR_P','GR','GR_N','GR_API','GAMMA_RAY','GAMMARAY','GRC','CGR','SGR','GR_RAW','GR_LOG')
LAS_DENS_NAMES=('DENS_P','RHOB','RHOBHC','DEN','DENS','RHOZ','ZDEN')
LAS_SONIC_NAMES=('DTC_P','DTC','DT','DTCO','AC')

def _las_curve(las,aliases):
    curves={re.sub(r'[^A-Z0-9]','',c.mnemonic.upper()):c.mnemonic for c in las.curves}
    for a in aliases:
        key=re.sub(r'[^A-Z0-9]','',a.upper())
        if key in curves:
            mnemonic=curves[key]
            return mnemonic,np.asarray(las[mnemonic],dtype=float)
    return None,None

def _las_depth_m(las):
    depth=np.asarray(las.index,dtype=float)
    index_mnemonic=str(las.index_unit or '').lower().strip()
    if index_mnemonic in ('ft','feet','foot'):depth=depth*0.3048
    elif index_mnemonic not in ('m','meter','metre','meters','metres'):
        raise ValueError(f'LAS depth unit not recognized: {las.index_unit!r}; verify manually')
    return depth

def load_las_logs(well,composite):
    try:import lasio
    except ImportError as exc:raise ImportError('LAS input required. Run %pip install lasio') from exc
    path=WELL_INPUT_PATHS[well]['las']
    if not path.is_file():raise FileNotFoundError(f'{well}: original LAS missing: {path}')
    las=lasio.read(str(path))
    md=_las_depth_m(las)
    # Composite MDKB_m is the mandatory depth bridge; prevents incorrect conversion
    # for deviated wells (LAS MD must never be treated as TVDSS).
    md_column=next((c for c in ('MDKB_m','MD_m','MD') if c in composite.columns),None)
    if md_column is None:
        raise ValueError(f'{well}: composite lacks MDKB_m / MD_m / MD. Cannot safely map LAS MD to TVDSS. '
                         'Supply the Phase 4 MD-to-TVDSS mapping, rather than assuming a vertical well.')
    bridge=composite[[md_column,'TVDSS_m']].copy()
    bridge[md_column]=pd.to_numeric(bridge[md_column],errors='coerce')
    bridge['TVDSS_m']=pd.to_numeric(bridge['TVDSS_m'],errors='coerce')
    bridge=bridge.dropna().sort_values(md_column).drop_duplicates(md_column)
    if len(bridge)<10:raise ValueError(f'{well}: insufficient valid MDKB-to-TVDSS pairs')
    bm=bridge[md_column].to_numpy(float);bz=bridge.TVDSS_m.to_numpy(float)
    if np.any(np.diff(bz)<-0.2):
        raise ValueError(f'{well}: MD-to-TVDSS bridge is nonmonotonic; inspect depth mapping')
    # Interpolate measured LAS curves onto composite's MD grid; preserve NaNs at edges.
    target_md=pd.to_numeric(composite[md_column],errors='coerce').to_numpy(float)
    info={'Well':well,'LAS_file':str(path),'Composite_file':str(WELL_INPUT_PATHS[well]['composite']),
          'LAS_depth_unit':str(las.index_unit),'Bridge_column':md_column}
    for output,aliases in [('GR_LAS',LAS_GR_NAMES),('RHOB_LAS',LAS_DENS_NAMES),('DTC_LAS',LAS_SONIC_NAMES)]:
        name,vals=_las_curve(las,aliases)
        info[output+'_mnemonic']=name or 'NOT FOUND'
        if vals is None:
            composite[output]=np.nan
            info[output+'_coverage']=0.
            continue
        good=np.isfinite(md)&np.isfinite(vals)
        if good.sum()<10:
            composite[output]=np.nan
            info[output+'_coverage']=0.
            continue
        x=md[good];v=vals[good];order=np.argsort(x);x=x[order];v=v[order]
        uniq=np.r_[True,np.diff(x)>0];x=x[uniq];v=v[uniq]
        composite[output]=np.interp(target_md,x,v,left=np.nan,right=np.nan)
        info[output+'_coverage']=float(np.mean(np.isfinite(composite[output])))
    # Source QC before enforcing the coverage threshold. Helps distinguish
    # missing mnemonic, empty curve and LAS/composite MD mismatch.
    info['LAS_MD_min_m']=float(np.nanmin(md))
    info['LAS_MD_max_m']=float(np.nanmax(md))
    info['Composite_MD_min_m']=float(np.nanmin(target_md))
    info['Composite_MD_max_m']=float(np.nanmax(target_md))
    info['MD_overlap_fraction']=float(np.mean(np.isfinite(target_md) &
        (target_md>=np.nanmin(md)) & (target_md<=np.nanmax(md))))
    print(f'{well} LAS QC: GR={info["GR_LAS_mnemonic"]}, '
          f'GR coverage={info["GR_LAS_coverage"]:.1%}, '
          f'MD overlap={info["MD_overlap_fraction"]:.1%}, '
          f'LAS MD={info["LAS_MD_min_m"]:.1f}–{info["LAS_MD_max_m"]:.1f} m, '
          f'composite MD={info["Composite_MD_min_m"]:.1f}–{info["Composite_MD_max_m"]:.1f} m')
    if info['GR_LAS_coverage']<MIN_LAS_GR_COVERAGE:
        raise ValueError(f'{well}: original LAS GR coverage {info["GR_LAS_coverage"]:.1%} is below '
                         f'{MIN_LAS_GR_COVERAGE:.0%}. Check LAS depth units, MD mapping and GR mnemonic. '
                         f'Available LAS curves: {[c.mnemonic for c in las.curves]}')
    return composite,info

def parse_marker_file(path):
    rows=[]
    for line in Path(path).read_text(errors='replace').splitlines():
        line=line.strip()
        if not line or line.startswith(('#','~',';')):continue
        parts=[x.strip().strip('"') for x in re.split(r'[\t,;]+|\s{2,}',line) if x.strip()]
        if len(parts)<2:parts=line.split()
        if len(parts)<2:continue
        if len(parts)>3:parts=[' '.join(parts[:-2]),parts[-2],parts[-1]]
        try:
            if len(parts)>=3:
                md,tvdss=float(parts[1]),float(parts[2])
            else:
                md,tvdss=np.nan,float(parts[1])
            if np.isfinite(tvdss):rows.append((parts[0].upper(),md,tvdss))
        except ValueError:pass
    return pd.DataFrame(rows,columns=['Original_Name','MDKB_m','TVDSS_m'])

def find_marker_file(well):
    """Discover each well's original marker file, including nested Well_To_Predict folders.

    Prefer files named for the exact well. Do not silently select ambiguous matches.
    """
    override=NEW_WELL_MARKER_OVERRIDES.get(well, WELL_MARKER_OVERRIDES.get(well))
    if override is not None and Path(override).is_file():
        return Path(override)
    number=well.split('_')[1]
    search_roots=[ROOT/'Well_To_Predict', ROOT/well, ROOT/f'Well-{number}']
    allowed={'.txt','.csv','.tsv'}
    def exact_well_token(text):
        return bool(re.search(rf'(?i)(?<![a-z0-9])well[_\- ]?0*{number}(?![0-9])',text))
    candidates=[]
    for base in search_roots:
        if not base.is_dir():continue
        for f in base.rglob('*'):
            if not f.is_file() or f.suffix.lower() not in allowed:continue
            if 'marker' not in f.stem.lower():continue
            rel=str(f.relative_to(base))
            if exact_well_token(rel) or (base.name.lower() in (well.lower(),f'well-{number}'.lower())):
                candidates.append(f)
    candidates=sorted(set(candidates))
    if not candidates:
        print(f'WARNING: {well} marker file not found under {search_roots}')
        return None
    if len(candidates)>1:
        # Favor a well-specific marker filename over marker files identified only by folder.
        named=[f for f in candidates if exact_well_token(f.stem)]
        if len(named)==1:
            return named[0]
        print(f'WARNING: multiple marker files found for {well}:')
        for f in candidates:print('  ',f)
        print(f'Set NEW_WELL_MARKER_OVERRIDES["{well}"] to the intended file.')
        return None
    return candidates[0]

# Explicit trajectory handling: use deviation when available, otherwise vertical.
# MD is measured from KB; TVDSS = TVD below KB minus KB elevation (metres).
def _norm_col(s):return re.sub(r'[^a-z0-9]','',str(s).lower())
def _find_col(tab,names):
    cols={_norm_col(c):c for c in tab.columns}
    return next((cols[_norm_col(n)] for n in names if _norm_col(n) in cols),None)
def _survey_table(path):
    """Read headerless APGCE deviation TXT (#MDKB, #TVD, #INCL, #AZIM)
    and conventional named-column survey/checkshot files."""
    path=Path(path)
    if path.suffix.lower()=='.las':
        import lasio
        return lasio.read(str(path)).df().reset_index()
    if 'deviation' in path.stem.lower() or 'trajectory' in path.stem.lower():
        lines=path.read_text(errors='replace').splitlines()
        header=' '.join(x.strip().lstrip('#').strip() for x in lines if x.lstrip().startswith('#')).upper()
        numeric=[line.strip() for line in lines if line.strip() and not line.lstrip().startswith('#')]
        if numeric:
            import io
            try:
                data=pd.read_csv(io.StringIO('\n'.join(numeric)),sep=r'\s+',header=None)
                if data.shape[1]==4 and all(pd.to_numeric(data[c],errors='coerce').notna().all() for c in data.columns):
                    # Only interpret unnamed four-column files when header confirms layout.
                    if all(key in header for key in ('MDKB','TVD','INCL','AZIM')):
                        data.columns=['MDKB_m','TVD_m','INCL_deg','AZIM_deg']
                        return data
            except Exception:
                pass
    for kw in ({'sep':None,'engine':'python','comment':'#'},
               {'sep':r'\s+','engine':'python','comment':'#'}):
        try:
            t=pd.read_csv(path,**kw)
            if len(t.columns)>=2:return t
        except Exception:pass
    raise ValueError(f'Cannot read tabular file: {path}')
def _find_well_file(well,kind,overrides):
    if well in overrides:
        f=Path(overrides[well]);
        if not f.is_file():raise FileNotFoundError(f)
        return f
    num=well.split('_')[1]
    matches=[]
    for f in TRAJECTORY_ROOT.rglob('*'):
        if not f.is_file() or f.suffix.lower() not in ('.csv','.txt','.tsv','.las'):continue
        name=str(f.relative_to(TRAJECTORY_ROOT))
        if not re.search(rf'(?i)(?<![a-z0-9])well[ _-]*0*{num}(?![0-9])',name):continue
        if (kind=='deviation' and any(k in f.stem.lower() for k in ('dev','trajectory','survey'))) or (kind=='checkshot' and any(k in f.stem.lower() for k in ('checkshot','check_shot'))):
            matches.append(f)
    if len(matches)>1:raise ValueError(f'{well}: ambiguous {kind}: {matches}. Set an override in Cell 1.')
    return matches[0] if matches else None
def _strict_interp(q,x,y):
    x=np.asarray(x,float);y=np.asarray(y,float);q=np.asarray(q,float)
    good=np.isfinite(x)&np.isfinite(y)
    x=x[good];y=y[good];order=np.argsort(x);x=x[order];y=y[order]
    keep=np.r_[True,np.diff(x)>0] if len(x) else np.array([],bool)
    x=x[keep];y=y[keep]
    if len(x)<2:raise ValueError('Not enough valid depth controls')
    return np.interp(q,x,y,left=np.nan,right=np.nan)
def _minimum_curvature_offsets(md,incl,azim):
    """North/east offsets in metres, azimuth clockwise from grid north."""
    md=np.asarray(md,float)
    inc=np.deg2rad(np.asarray(incl,float))
    azi=np.unwrap(np.deg2rad(np.asarray(azim,float)))
    east=np.zeros(len(md));north=np.zeros(len(md))
    for k in range(1,len(md)):
        dmd=md[k]-md[k-1]
        dogleg=np.arccos(np.clip(np.cos(inc[k-1])*np.cos(inc[k])+
            np.sin(inc[k-1])*np.sin(inc[k])*np.cos(azi[k]-azi[k-1]),-1,1))
        rf=1. if dogleg<1e-9 else 2*np.tan(dogleg/2)/dogleg
        east[k]=east[k-1]+dmd/2*(np.sin(inc[k-1])*np.sin(azi[k-1])+
                                          np.sin(inc[k])*np.sin(azi[k]))*rf
        north[k]=north[k-1]+dmd/2*(np.sin(inc[k-1])*np.cos(azi[k-1])+
                                            np.sin(inc[k])*np.cos(azi[k]))*rf
    return east,north
def _load_trajectory(well):
    f=_find_well_file(well,'deviation',DEVIATION_FILE_OVERRIDES)
    if f is None:return {'mode':'VERTICAL','file':None}
    t=_survey_table(f)
    md=_find_col(t,['MDKB_m','MD_m','MD','MeasuredDepth','Measured_Depth','Depth_MD'])
    tvdss=_find_col(t,['TVDSS_m','TVDSS','TVD_SS'])
    tvd=_find_col(t,['TVD_m','TVD','TrueVerticalDepth'])
    x=_find_col(t,['X','X_m','Easting','Easting_m','UTM_X'])
    y=_find_col(t,['Y','Y_m','Northing','Northing_m','UTM_Y'])
    incl=_find_col(t,['INCL_deg','INCL','Inclination','Inc'])
    azim=_find_col(t,['AZIM_deg','AZIM','Azimuth','Azi'])
    if md is None or (tvdss is None and tvd is None):
        raise ValueError(f'{well}: deviation {f} must contain MD and TVD/TVDSS. Found {list(t.columns)}')
    for col in [md,tvdss,tvd,x,y]:
        if col:t[col]=pd.to_numeric(t[col],errors='coerce')
    t=t.dropna(subset=[md,tvdss or tvd]).sort_values(md).drop_duplicates(md)
    if len(t)<2:raise ValueError(f'{well}: insufficient deviation stations')
    depth=t[tvdss].to_numpy(float) if tvdss else t[tvd].to_numpy(float)-KB_M[well]
    if np.any(np.diff(depth)<-0.1):raise ValueError(f'{well}: nonmonotonic trajectory TVDSS')
    if x is not None and y is not None:
        east=t[x].to_numpy(float);north=t[y].to_numpy(float)
        xy_source='SURVEY_ABSOLUTE'
    elif incl is not None and azim is not None:
        east_offset,north_offset=_minimum_curvature_offsets(
            t[md].to_numpy(float),
            pd.to_numeric(t[incl],errors='coerce').to_numpy(float),
            pd.to_numeric(t[azim],errors='coerce').to_numpy(float))
        whx,why=_wellhead_xy(well)
        east=east_offset+whx if np.isfinite(whx) else None
        north=north_offset+why if np.isfinite(why) else None
        xy_source='MIN_CURVATURE_WITH_WELLHEAD' if east is not None and north is not None else 'MIN_CURVATURE_OFFSETS_ONLY'
    else:
        east=north=None
        xy_source='NO_XY_OR_INCL_AZIM'
    print(f'{well}: deviation loaded ({len(t)} stations), XY={xy_source}')
    return {'mode':'DEVIATED','file':f,'MD':t[md].to_numpy(float),'TVDSS':depth,
            'X':east,'Y':north,'XY_SOURCE':xy_source}

def _composite_md_column(df):return _find_col(df,['MDKB_m','MD_m','MD','MeasuredDepth'])
def _wellhead_xy(well):
    """Read absolute wellhead coordinates from original LAS headers, not log curves."""
    if well in WELLHEAD_XY_OVERRIDES:
        return tuple(map(float,WELLHEAD_XY_OVERRIDES[well]))
    import lasio
    las_path=WELL_INPUT_PATHS[well]['las']
    if not las_path.is_file():
        raise FileNotFoundError(f'{well}: LAS file not found: {las_path}')
    las=lasio.read(str(las_path))
    # LAS headers may store the same field under several conventional names.
    x_names={'X','XCOORD','XCOORDINATE','EASTING','UTMX','WELLHEADX','SURFACEX','SURFACEEASTING'}
    y_names={'Y','YCOORD','YCOORDINATE','NORTHING','UTMY','WELLHEADY','SURFACEY','SURFACENORTHING'}
    found={'X':[],'Y':[]}
    for section_name in ('well','params','other'):
        section=getattr(las,section_name,None)
        if section is None:continue
        if section_name=='other':
            # Free-text header lines are not parsed as coordinates without a known format.
            continue
        for item in section:
            key=re.sub(r'[^A-Z0-9]','',str(item.mnemonic).upper())
            if key not in x_names and key not in y_names:continue
            raw=str(item.value).strip().replace(',','')
            try:value=float(raw)
            except ValueError:continue
            if not np.isfinite(value):continue
            axis='X' if key in x_names else 'Y'
            found[axis].append((section_name,item.mnemonic,value))
    if not found['X'] or not found['Y']:
        print(f'{well}: LAS header wellhead X/Y not recognized; '
              f'available WELL mnemonics={[v.mnemonic for v in las.well]}; '
              f'PARAM mnemonics={[v.mnemonic for v in las.params]}')
        return np.nan,np.nan
    xs=np.array([v for _,_,v in found['X']]);ys=np.array([v for _,_,v in found['Y']])
    if np.ptp(xs)>1 or np.ptp(ys)>1:
        raise ValueError(f'{well}: conflicting LAS header coordinates: {found}. '
                         'Specify WELLHEAD_XY_OVERRIDES after verifying the CRS.')
    x,y=float(xs[0]),float(ys[0])
    print(f'{well}: LAS wellhead X={x:.3f}, Y={y:.3f} '
          f'(fields {found["X"][0][1]}, {found["Y"][0][1]}; verify CRS/units)')
    return x,y

TRAJECTORIES={well:_load_trajectory(well) for well in WELL_NAMES}
TRAJECTORY_QC=[]
def _apply_trajectory_to_composite(well,df):
    tr=TRAJECTORIES[well]
    mdcol=_composite_md_column(df)
    if mdcol is None:raise ValueError(f'{well}: composite missing MD. Need MD for LAS and trajectory mapping.')
    md=pd.to_numeric(df[mdcol],errors='coerce').to_numpy(float)
    df['MDKB_m']=md
    if tr['mode']=='DEVIATED':
        z=_strict_interp(md,tr['MD'],tr['TVDSS'])
    else:
        z=md-KB_M[well]
    if 'TVDSS_m' in df:
        original=pd.to_numeric(df['TVDSS_m'],errors='coerce').to_numpy(float)
        good=np.isfinite(z)&np.isfinite(original)
        mismatch=float(np.nanmedian(np.abs(z[good]-original[good]))) if good.any() else np.nan
        if np.isfinite(mismatch) and mismatch>MAX_COMPOSITE_TVDSS_MISMATCH_M:
            raise ValueError(f'{well}: composite vs deviation-derived TVDSS median mismatch '
                             f'{mismatch:.2f} m. Check MD datum, KB, survey units before correlation.')
    df['TVDSS_m']=z
    good=np.isfinite(md)&np.isfinite(z)
    if good.sum()<20:raise ValueError(f'{well}: fewer than 20 valid trajectory-converted TVDSS samples')
    if np.any(np.diff(z[good])<-0.1):raise ValueError(f'{well}: trajectory TVDSS reverses')
    TRAJECTORY_QC.append({'Well':well,'Mode':tr['mode'],'Deviation':str(tr['file']),
                          'TVDSS_Coverage':float(good.mean())})
    return df

WELL_DATA={}; SOURCE_STATUS=[]; MARKER_ROWS=[]; LAS_SOURCE_QC=[]
for well in WELL_NAMES:
    result_path=pick_result_file(well)
    if result_path is None:
        SOURCE_STATUS.append({'Well':well,'Result':'MISSING','Marker':'NOT_CHECKED'})
        continue
    df=pd.read_csv(result_path,low_memory=False)
    for col in df.columns:
        if col not in ('LITHOLOGY','TD_SOURCE'):
            df[col]=pd.to_numeric(df[col],errors='coerce')
    df=_apply_trajectory_to_composite(well,df)
    df=df.dropna(subset=['TVDSS_m']).sort_values('TVDSS_m').drop_duplicates('TVDSS_m')
    if len(df)<20:raise ValueError(f'{well}: too few valid depth samples')
    df,las_info=load_las_logs(well,df.reset_index(drop=True))
    LAS_SOURCE_QC.append(las_info)
    WELL_DATA[well]=df
    marker_path=find_marker_file(well)
    if marker_path is not None and marker_path.is_file():
        markers=parse_marker_file(marker_path)
        for _,r in markers.iterrows():
            key=(well,r.Original_Name)
            if key not in ALIASES:continue # not in approved matrix or excluded
            MARKER_ROWS.append({'Well':well,'Canonical':ALIASES[key],
                                'Original_Name':r.Original_Name,
                                'TVDSS_m':(float(_strict_interp([r.MDKB_m],TRAJECTORIES[well]['MD'],TRAJECTORIES[well]['TVDSS'])[0]) if TRAJECTORIES[well]['mode']=='DEVIATED' and np.isfinite(r.MDKB_m) else (float(r.MDKB_m-KB_M[well]) if TRAJECTORIES[well]['mode']=='VERTICAL' and np.isfinite(r.MDKB_m) else float(r.TVDSS_m))),
                                'MDKB_m':r.MDKB_m,'Source_File':str(marker_path),
                                'Type':'MEASURED'})
        approved_count=sum(x['Well']==well for x in MARKER_ROWS)
        marker_status=f'{len(markers)} raw; {approved_count} approved'
        print(f'{well} marker source: {marker_path} ({marker_status})')
        if len(markers)>0 and approved_count==0:
            print(f'WARNING: {well}: file parsed, but no marker names matched the approved Step 5 matrix.')
            print('  First names:',markers.Original_Name.head(8).tolist())
    else:
        marker_status='NO_FILE'
    SOURCE_STATUS.append({'Well':well,'Result':str(result_path),'Marker':marker_status})
SOURCE_STATUS=pd.DataFrame(SOURCE_STATUS)
MEASURED=pd.DataFrame(MARKER_ROWS,columns=['Well','Canonical','Original_Name',
                                          'TVDSS_m','MDKB_m','Source_File','Type'])
SOURCE_STATUS.to_csv(OUTPUT_ROOT/'Phase5_Source_Status.csv',index=False)
LAS_SOURCE_QC=pd.DataFrame(LAS_SOURCE_QC)
LAS_SOURCE_QC.to_csv(OUTPUT_ROOT/'Phase5_Original_LAS_Composite_Source_QC.csv',index=False)
print('Original LAS and predicted composite source QC:')
TRAJECTORY_QC=pd.DataFrame(TRAJECTORY_QC)
display(TRAJECTORY_QC)
TRAJECTORY_QC.to_csv(OUTPUT_ROOT/'Phase5_Trajectory_Depth_QC.csv',index=False)
display(LAS_SOURCE_QC)
display(SOURCE_STATUS)
if REFERENCE_WELL not in WELL_DATA:raise RuntimeError('Reference well unavailable; set WELL_RESULT_OVERRIDES')
if not any((MEASURED.Well==REFERENCE_WELL)&(MEASURED.Canonical==REGIONAL_COAL_ANCHOR)):
    raise RuntimeError('Reference I-50_COAL marker missing. Verify Well 4 marker file and canonical names.')
if len(WELL_DATA)<2:raise RuntimeError('Need at least two wells with prediction/lithology CSVs')

print("Cell 3 completed")


Well_2: LAS wellhead X=471455.520, Y=573417.890 (fields X, Y; verify CRS/units)
Well_2: deviation loaded (16 stations), XY=MIN_CURVATURE_WITH_WELLHEAD
Well_4: LAS wellhead X=470944.510, Y=575660.470 (fields X, Y; verify CRS/units)
Well_4: deviation loaded (14 stations), XY=MIN_CURVATURE_WITH_WELLHEAD
Well_5: LAS wellhead X=462919.180, Y=575759.860 (fields X, Y; verify CRS/units)
Well_5: deviation loaded (21 stations), XY=MIN_CURVATURE_WITH_WELLHEAD
Well_6: LAS wellhead X=463912.300, Y=570036.450 (fields X, Y; verify CRS/units)
Well_6: deviation loaded (18 stations), XY=MIN_CURVATURE_WITH_WELLHEAD
Well_1 LAS QC: GR=GR_P, GR coverage=100.0%, MD overlap=100.0%, LAS MD=335.0–3072.4 m, composite MD=335.0–3072.4 m
Well_1 marker source: /home/apgce_team_08/link-to-apgce_team_08/Well_To_Predict/Well-1_Markers.txt (5 raw; 5 approved)
Well_2 LAS QC: GR=GR_P, GR coverage=99.4%, MD overlap=100.0%, LAS MD=352.2–3395.9 m, composite MD=352.2–3395.9 m
Well_2 marker source: /home/apgce_team_08/link-to-

,Well,Mode,Deviation,TVDSS_Coverage
0,Well_1,VERTICAL,None,1.000000
1,Well_2,DEVIATED,/home/apgce_team_08/To_Overlay_Horizon/Well-2_...,1.000000
2,Well_4,DEVIATED,/home/apgce_team_08/To_Overlay_Horizon/Well-4_...,0.990255
3,Well_5,DEVIATED,/home/apgce_team_08/To_Overlay_Horizon/Well-5_...,0.987098
4,Well_6,DEVIATED,/home/apgce_team_08/To_Overlay_Horizon/Well-6_...,1.000000
5,Well_7,VERTICAL,None,1.000000
6,Well_8,VERTICAL,None,1.000000


,Well,LAS_file,Composite_file,LAS_depth_unit,Bridge_column,GR_LAS_mnemonic,GR_LAS_coverage,RHOB_LAS_mnemonic,RHOB_LAS_coverage,DTC_LAS_mnemonic,DTC_LAS_coverage,LAS_MD_min_m,LAS_MD_max_m,Composite_MD_min_m,Composite_MD_max_m,MD_overlap_fraction
0,Well_1,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,0.999666,DENS_P,0.764906,DTC_P,0.998163,335.000000,3072.408700,335.0000,3072.4087,1.0
1,Well_2,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,0.993792,DENS_P,0.991238,NOT FOUND,0.000000,352.196400,3395.929200,352.1964,3395.9292,1.0
2,Well_4,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,0.980220,DENS_P,0.980415,DTC_P,0.990354,1089.964759,2669.286133,1089.9648,2653.8937,1.0
3,Well_5,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,1.000000,DENS_P,0.966860,DTC_P,0.966860,1061.770800,2230.983600,1061.7708,2215.8960,1.0
4,Well_6,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,1.000000,DENS_P,1.000000,NOT FOUND,0.000000,1021.080000,2236.317600,1021.0800,2209.9524,1.0
5,Well_7,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,1.000000,DENS_P,1.000000,NOT FOUND,0.000000,1002.030000,2132.533200,1002.0300,2132.5332,1.0
6,Well_8,/home/apgce_team_08/link-to-apgce_team_08/Well...,/home/apgce_team_08/link-to-apgce_team_08/Well...,M,MDKB_m,GR_P,0.975563,DENS_P,0.957085,DTC_P,1.000000,957.376800,1979.980800,957.3768,1979.9808,1.0


,Well,Result,Marker
0,Well_1,/home/apgce_team_08/link-to-apgce_team_08/Well...,5 raw; 5 approved
1,Well_2,/home/apgce_team_08/link-to-apgce_team_08/Well...,23 raw; 22 approved
2,Well_4,/home/apgce_team_08/link-to-apgce_team_08/Well...,22 raw; 21 approved
3,Well_5,/home/apgce_team_08/link-to-apgce_team_08/Well...,16 raw; 16 approved
4,Well_6,/home/apgce_team_08/link-to-apgce_team_08/Well...,15 raw; 15 approved
5,Well_7,/home/apgce_team_08/link-to-apgce_team_08/Well...,7 raw; 7 approved
6,Well_8,/home/apgce_team_08/link-to-apgce_team_08/Well...,11 raw; 11 approved


Cell 3 completed


In [4]:
# CELL 4 — Mandatory checkshot-based conversion of ALL curves and markers to TWT
# Each log is already trajectory-corrected to TVDSS in Cell 3.
# Checkshot depth may be MD, TVD, or TVDSS; conversion is explicit.
TIME_SOURCES={}; TIME_QC=[]
def _read_checkshot(well,path):
    # APGCE checkshot format: comment headers followed by two numeric columns.
    # Parse the declared depth/time basis; never infer it from numerical magnitudes.
    import io, re
    path=Path(path)
    lines=path.read_text(errors='replace').splitlines()
    header_lines=[line.lstrip().lstrip('#').strip() for line in lines if line.lstrip().startswith('#')]
    header=' '.join(header_lines).upper()
    numeric_lines=[line.strip() for line in lines if line.strip() and not line.lstrip().startswith('#')]
    tab=None
    if numeric_lines and 'CHECKSHOT' in header:
        try:
            raw=pd.read_csv(io.StringIO('\n'.join(numeric_lines)),sep=r'\s+',header=None)
            if raw.shape[1]==2 and raw.apply(lambda c:pd.to_numeric(c,errors='coerce').notna().all()).all():
                if 'TWT' not in header or not re.search(r'TWT\s*\(\s*MS\s*\)',header):
                    raise ValueError(f'{well}: checkshot {path} must explicitly specify TWT (ms)')
                if 'TVDSS' in header:
                    raw.columns=['TVDSS_m','TWT_ms']
                elif re.search(r'\bTVD\b',header):
                    raw.columns=['TVD_m','TWT_ms']
                elif 'MDKB' in header:
                    raw.columns=['MDKB_m','TWT_ms']
                else:
                    raise ValueError(f'{well}: cannot establish checkshot depth basis from header: {header_lines}')
                tab=raw
                print(f'{well}: parsed headerless checkshot {path.name} as {list(tab.columns)}; {len(tab)} rows')
        except (pd.errors.ParserError,TypeError) as exc:
            raise ValueError(f'{well}: malformed checkshot {path}: {exc}') from exc
    if tab is None:
        tab=_survey_table(path)

    mdcol=_find_col(tab,['MDKB_m','MD_m','MD','MeasuredDepth','Measured_Depth'])
    zcol=_find_col(tab,['TVDSS_m','TVDSS','TVD_SS'])
    tvdcol=_find_col(tab,['TVD_m','TVD','TrueVerticalDepth'])
    tcol=_find_col(tab,['TWT_ms','TWT','TwoWayTime_ms','Two_Way_Time_ms','Time_2Way_ms'])
    if tcol is None:raise ValueError(f'{well}: checkshot {path} has no identifiable TWT column: {list(tab.columns)}')
    if zcol:
        z=pd.to_numeric(tab[zcol],errors='coerce').to_numpy(float)
        depth_source='TVDSS'
    elif tvdcol:
        z=pd.to_numeric(tab[tvdcol],errors='coerce').to_numpy(float)-KB_M[well]
        depth_source='TVD_MINUS_KB'
    elif mdcol:
        md=pd.to_numeric(tab[mdcol],errors='coerce').to_numpy(float)
        tr=TRAJECTORIES[well]
        z=_strict_interp(md,tr['MD'],tr['TVDSS']) if tr['mode']=='DEVIATED' else md-KB_M[well]
        depth_source='MD_VIA_TRAJECTORY'
    else:raise ValueError(f'{well}: checkshot has no MD/TVD/TVDSS column: {list(tab.columns)}')
    twt=pd.to_numeric(tab[tcol],errors='coerce').to_numpy(float)
    good=np.isfinite(z)&np.isfinite(twt)
    z=z[good];twt=twt[good];order=np.argsort(z);z=z[order];twt=twt[order]
    keep=np.r_[True,np.diff(z)>0] if len(z) else np.array([],bool)
    z=z[keep];twt=twt[keep]
    if len(z)<2 or np.any(np.diff(twt)<=0):raise ValueError(f'{well}: invalid/nonmonotonic checkshot')
    # No automatic seconds/milliseconds guessing; time column must declare ms.
    if 'ms' not in _norm_col(tcol) and _norm_col(tcol) not in ('twt',):
        raise ValueError(f'{well}: verify time units for checkshot column {tcol!r}')
    return z,twt,depth_source

if 'TWT_ms' not in MEASURED.columns:MEASURED['TWT_ms']=np.nan
for well in WELL_NAMES:
    shot=_find_well_file(well,'checkshot',CHECKSHOT_OVERRIDES)
    if shot is None:raise FileNotFoundError(f'{well}: checkshot missing under {TRAJECTORY_ROOT}. No correlation performed.')
    zd,td,source=_read_checkshot(well,shot)
    df=WELL_DATA[well]
    df['TWT_ms']=_strict_interp(df['TVDSS_m'],zd,td)
    good=np.isfinite(df['TWT_ms'].to_numpy(float))
    if good.mean()<MIN_VALID_TWT_COVERAGE:
        raise ValueError(f'{well}: checkshot TWT coverage {good.mean():.1%} below {MIN_VALID_TWT_COVERAGE:.0%}')
    df=df.sort_values('TVDSS_m').reset_index(drop=True)
    tw=df.TWT_ms.to_numpy(float);valid=tw[np.isfinite(tw)]
    if np.any(np.diff(valid)<=0):raise ValueError(f'{well}: log TWT not strictly increasing')
    WELL_DATA[well]=df
    idx=MEASURED.index[MEASURED.Well==well]
    if len(idx):MEASURED.loc[idx,'TWT_ms']=_strict_interp(MEASURED.loc[idx,'TVDSS_m'],zd,td)
    TIME_SOURCES[well]=str(shot)
    TIME_QC.append({'Well':well,'Checkshot':str(shot),'Depth_basis':source,
                    'TWT_Coverage':float(good.mean()),'TWT_min_ms':float(valid.min()),
                    'TWT_max_ms':float(valid.max()),'Trajectory':TRAJECTORIES[well]['mode']})
TIME_QC=pd.DataFrame(TIME_QC)
TIME_QC.to_csv(OUTPUT_ROOT/'Phase5_PreCorrelation_TVDSS_TWT_QC.csv',index=False)
display(TIME_QC)
MEASURED=MEASURED.dropna(subset=['TWT_ms']).copy()
print('All seven wells: trajectory-corrected TVDSS → checkshot TWT completed BEFORE correlation.')
print('Cell 4 completed')

Well_1: parsed headerless checkshot Well-1_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 39 rows
Well_2: parsed headerless checkshot Well-2_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 31 rows
Well_4: parsed headerless checkshot Well-4_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 29 rows
Well_5: parsed headerless checkshot Well-5_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 17 rows
Well_6: parsed headerless checkshot Well-6_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 24 rows
Well_7: parsed headerless checkshot Well-7_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 20 rows
Well_8: parsed headerless checkshot Well-8_Checkshot.txt as ['TVDSS_m', 'TWT_ms']; 38 rows


,Well,Checkshot,Depth_basis,TWT_Coverage,TWT_min_ms,TWT_max_ms,Trajectory
0,Well_1,/home/apgce_team_08/To_Overlay_Horizon/Well-1_...,TVDSS,0.997161,386.373491,2435.931083,VERTICAL
1,Well_2,/home/apgce_team_08/To_Overlay_Horizon/Well-2_...,TVDSS,0.970961,441.519229,2538.386553,DEVIATED
2,Well_4,/home/apgce_team_08/To_Overlay_Horizon/Well-4_...,TVDSS,0.957225,1151.784177,2158.418822,DEVIATED
3,Well_5,/home/apgce_team_08/To_Overlay_Horizon/Well-5_...,TVDSS,0.943359,1163.243376,1932.583087,DEVIATED
4,Well_6,/home/apgce_team_08/To_Overlay_Horizon/Well-6_...,TVDSS,0.995001,1067.314916,1909.968971,DEVIATED
5,Well_7,/home/apgce_team_08/To_Overlay_Horizon/Well-7_...,TVDSS,0.933684,1086.209450,1846.777718,VERTICAL
6,Well_8,/home/apgce_team_08/To_Overlay_Horizon/Well-8_...,TVDSS,0.985695,1018.559840,1768.099422,VERTICAL


All seven wells: trajectory-corrected TVDSS → checkshot TWT completed BEFORE correlation.
Cell 4 completed


In [5]:
# CELL 5 — Phase 5 TWT well correlation
# Identify coal using Phase 2 lithology when available; otherwise FCOAL
# probability; GR/RHOB fallback is explicitly low-confidence.
COAL_INTERVALS=[]
COAL_DIAGNOSTICS=[]
COAL_SOURCE={}
for well,df in WELL_DATA.items():
    z=df.TVDSS_m.to_numpy(float)
    dz=np.nanmedian(np.diff(z))
    # Never assume lithology code 1 is coal: code mappings vary by phase.
    # Prefer explicitly named coal fraction / probability where available.
    available=[c for c in ('FCOAL','COAL_PROBABILITY','P_COAL','COAL_FRAC')
               if c in df and pd.to_numeric(df[c],errors='coerce').notna().any()]
    if available:
        col=available[0]
        coal=np.clip(pd.to_numeric(df[col],errors='coerce').to_numpy(float),0,1)
        method=f'Explicit coal indicator: {col}'
    elif 'LITHOLOGY' in df and df.LITHOLOGY.notna().any():
        labels=df.LITHOLOGY.astype(str).str.upper().str.strip()
        coal=labels.str.contains(r'\bCOAL\b',regex=True).to_numpy(float)
        method='Explicit COAL lithology label'
    elif {'GR_P','DENS_P'}.issubset(df.columns):
        coal=((df.GR_P.to_numpy(float)<COAL_GR_CUTOFF)&
              (df.DENS_P.to_numpy(float)<COAL_RHOB_CUTOFF)).astype(float)
        method='GR/RHOB heuristic — LOW CONFIDENCE'
    else:
        warnings.warn(f'{well}: no verified coal indicator; inspect LITHOLOGY_CODE mapping')
        COAL_SOURCE[well]='UNAVAILABLE (numeric lithology code not assumed)'
        continue
    COAL_SOURCE[well]=method
    mask=np.isfinite(coal)&(coal>=COAL_PROBABILITY_THRESHOLD)
    COAL_DIAGNOSTICS.append({'Well':well,'Method':method,
        'Samples':len(df),'Coal_fraction_samples':int(mask.sum()),
        'TWT_coverage_pct':round(100*np.isfinite(df.TWT_ms).mean(),2),
        'Lithology_codes':str(df.LITHOLOGY_CODE.value_counts().head(8).to_dict())
                           if 'LITHOLOGY_CODE' in df else 'N/A'})
    # Bridge short internal gaps, but do not fill long intervals.
    starts=np.flatnonzero(np.diff(np.r_[False,mask,False].astype(int))==1)
    ends=np.flatnonzero(np.diff(np.r_[False,mask,False].astype(int))==-1)
    for end,begin in zip(ends[:-1],starts[1:]):
        if z[begin]-z[end-1] <= COAL_GAP_BRIDGE_M:
            mask[end:begin]=True
    starts=np.flatnonzero(np.diff(np.r_[False,mask,False].astype(int))==1)
    ends=np.flatnonzero(np.diff(np.r_[False,mask,False].astype(int))==-1)
    for a,b in zip(starts,ends):
        top=float(z[a]);base=float(z[b-1]+max(dz,0))
        thick=base-top
        if thick>MIN_COAL_THICKNESS_M:
            top_twt=float(np.interp(top,z,df.TWT_ms.to_numpy(float)))
            # At the bottom edge, use the final valid sample time; do not
            # discard the bed because base extends by half a sample.
            base_time_depth=min(base,float(z[-1]))
            base_twt=float(np.interp(base_time_depth,z,df.TWT_ms.to_numpy(float),
                                     left=np.nan,right=np.nan))
            if not np.isfinite(top_twt) or not np.isfinite(base_twt):continue
            COAL_INTERVALS.append({'Well':well,'Top_TWT_ms':top_twt,
                 'Base_TWT_ms':base_twt,'Mid_TWT_ms':(top_twt+base_twt)/2,
                 'Top_TVDSS_m':top,
                 'Base_TVDSS_m':base,'Mid_TVDSS_m':(top+base)/2,
                 'Thickness_m':thick,'Mean_Coal_Score':float(np.nanmean(coal[a:b])),
                 'Detection_Method':method})
COALS=pd.DataFrame(COAL_INTERVALS)
COAL_DIAGNOSTICS=pd.DataFrame(COAL_DIAGNOSTICS)
COAL_DIAGNOSTICS.to_csv(OUTPUT_ROOT/'Phase5_Coal_Detection_Diagnostics.csv',index=False)
display(COAL_DIAGNOSTICS)
if COALS.empty:
    warnings.warn(f'No verified >{MIN_COAL_THICKNESS_M:g} m coal beds detected. Marker-only correlation '
                  'will continue; thick-coal correlations are withheld.')
    COALS=pd.DataFrame(columns=['Well','Top_TWT_ms','Base_TWT_ms','Mid_TWT_ms',
        'Top_TVDSS_m','Base_TVDSS_m','Mid_TVDSS_m','Thickness_m',
        'Mean_Coal_Score','Detection_Method'])
COALS.to_csv(OUTPUT_ROOT/'Phase5_Detected_Coals_Over_Threshold.csv',index=False)
print('Coal detection methods:',COAL_SOURCE)
if not COALS.empty:
    display(COALS.groupby('Well').agg(Beds=('Thickness_m','size'),
                                      Max_thickness_m=('Thickness_m','max')))
else:
    print('No qualifying thick coals; inspect diagnostic CSV before interpreting coal ties.')

print("Cell 5 completed")


,Well,Method,Samples,Coal_fraction_samples,TWT_coverage_pct,Lithology_codes
0,Well_1,Explicit coal indicator: FCOAL,17963,514,99.72,N/A
1,Well_2,Explicit coal indicator: FCOAL,19973,145,97.10,N/A
2,Well_4,Explicit coal indicator: FCOAL,10263,157,95.72,N/A
3,Well_5,Explicit coal indicator: FCOAL,7574,128,94.34,N/A
4,Well_6,Explicit coal indicator: FCOAL,7802,130,99.50,N/A
5,Well_7,Explicit coal indicator: FCOAL,7419,173,93.37,N/A
6,Well_8,Explicit coal indicator: FCOAL,6711,116,98.57,N/A


Coal detection methods: {'Well_1': 'Explicit coal indicator: FCOAL', 'Well_2': 'Explicit coal indicator: FCOAL', 'Well_4': 'Explicit coal indicator: FCOAL', 'Well_5': 'Explicit coal indicator: FCOAL', 'Well_6': 'Explicit coal indicator: FCOAL', 'Well_7': 'Explicit coal indicator: FCOAL', 'Well_8': 'Explicit coal indicator: FCOAL'}


,Beds,Max_thickness_m
Well,,
Well_1,18,3.810000
Well_2,1,11.733415
Well_4,3,1.523991
Well_5,1,3.045888
Well_6,1,1.828800
Well_7,2,2.590800
Well_8,1,1.676400


Cell 5 completed


In [6]:
# CELL 6 — Phase 5 TWT well correlation
# Build first-pass regional depth shifts from measured I-50_COAL anchors.
# Where absent, use robust median of *shared approved markers* only as a
# provisional offset; mark such wells as lower confidence.
ANCHOR=MEASURED[MEASURED.Canonical==REGIONAL_COAL_ANCHOR]
ref_anchor=float(ANCHOR.loc[ANCHOR.Well==REFERENCE_WELL,'TWT_ms'].median())
REGIONAL_OFFSETS={}; ANCHOR_METHOD={}
for well in WELL_DATA:
    direct=ANCHOR[ANCHOR.Well==well]
    if not direct.empty:
        REGIONAL_OFFSETS[well]=float(direct.TWT_ms.median()-ref_anchor)
        ANCHOR_METHOD[well]='MEASURED_I50_COAL'
        continue
    shared=MEASURED[MEASURED.Well.isin([well,REFERENCE_WELL])]
    pivot=shared.pivot_table(index='Canonical',columns='Well',
                             values='TWT_ms',aggfunc='median')
    if well in pivot and REFERENCE_WELL in pivot:
        both=pivot[[well,REFERENCE_WELL]].dropna()
        if len(both)>=2:
            REGIONAL_OFFSETS[well]=float(np.median(both[well]-both[REFERENCE_WELL]))
            ANCHOR_METHOD[well]=f'PROVISIONAL_{len(both)}_SHARED_MARKERS'
            continue
    REGIONAL_OFFSETS[well]=np.nan
    ANCHOR_METHOD[well]='UNANCHORED'
print('Regional depth offsets (m):')
display(pd.DataFrame({'Well':list(REGIONAL_OFFSETS),
                      'Offset_m':list(REGIONAL_OFFSETS.values()),
                      'Anchor_method':list(ANCHOR_METHOD.values())}))

print("Cell 6 completed")


Regional depth offsets (m):


,Well,Offset_m,Anchor_method
0,Well_1,6.549963,PROVISIONAL_3_SHARED_MARKERS
1,Well_2,33.683221,PROVISIONAL_9_SHARED_MARKERS
2,Well_4,0.000000,MEASURED_I50_COAL
3,Well_5,30.441111,PROVISIONAL_11_SHARED_MARKERS
4,Well_6,10.847628,PROVISIONAL_11_SHARED_MARKERS
5,Well_7,20.538163,MEASURED_I50_COAL
6,Well_8,57.815108,PROVISIONAL_4_SHARED_MARKERS


Cell 6 completed


In [7]:
# CELL 7 — Phase 5 TWT well correlation
# Match thick coals by minimum-cost bipartite assignment, relative to the
# regional I-50 anchor. Unanchored wells are NOT automatically matched.
# Seed each regional coal name from Well 4 thick coal beds.
ref_coals=COALS[COALS.Well==REFERENCE_WELL].sort_values('Top_TWT_ms').copy()
if ref_coals.empty:
    print(f'No other verified >{MIN_COAL_THICKNESS_M:g} m coal in {REFERENCE_WELL}; '
          'I-50_COAL remains a valid regional marker regardless of thickness.')
ref_coals['Regional_Name']=[f'REG_COAL_{i:02d}' for i in range(1,len(ref_coals)+1)]
# I-50_COAL is EXEMPT from minimum coal thickness: it remains a measured
# regional marker even if no coal interval is detected. Other coal beds must
# satisfy MIN_COAL_THICKNESS_M. Never fabricate a bed around the marker.
# If an eligible detected bed contains I-50, reuse its established name.
i50_ref=ref_coals[(ref_coals.Top_TWT_ms<=ref_anchor)&
                   (ref_coals.Base_TWT_ms>=ref_anchor)]
if len(i50_ref):
    ref_coals.loc[i50_ref.index[0],'Regional_Name']=REGIONAL_COAL_ANCHOR
else:
    print('I-50_COAL: thickness exemption applied. Existing marker TWT is retained '
          'as the regional anchor; no coal-bed top/base is fabricated.')
coal_matches=[]
for _,r in ref_coals.iterrows():
    coal_matches.append({'Regional_Name':r.Regional_Name,'Well':REFERENCE_WELL,
                         'Top_TWT_ms':r.Top_TWT_ms,'Base_TWT_ms':r.Base_TWT_ms,
                         'Thickness_m':r.Thickness_m,'Residual_m':0.,
                         'Confidence':'REFERENCE'})
for well in WELL_DATA:
    if well==REFERENCE_WELL or not np.isfinite(REGIONAL_OFFSETS[well]):continue
    targets=COALS[COALS.Well==well].reset_index(drop=True)
    if targets.empty:continue
    ref=ref_coals.reset_index(drop=True)
    residual=np.abs((targets.Mid_TWT_ms.to_numpy()[None,:]-
                     REGIONAL_OFFSETS[well])-ref.Mid_TWT_ms.to_numpy()[:,None])
    thickness=np.abs(np.log(np.maximum(targets.Thickness_m.to_numpy()[None,:],.1)/
                            np.maximum(ref.Thickness_m.to_numpy()[:,None],.1)))
    cost=residual+5*thickness
    rows,cols=linear_sum_assignment(cost)
    for a,b in zip(rows,cols):
        if residual[a,b]>MAX_COAL_MATCH_RESIDUAL_MS:continue
        r,t=ref.iloc[a],targets.iloc[b]
        coal_matches.append({'Regional_Name':r.Regional_Name,'Well':well,
                             'Top_TWT_ms':t.Top_TWT_ms,'Base_TWT_ms':t.Base_TWT_ms,
                             'Thickness_m':t.Thickness_m,
                             'Residual_m':float(residual[a,b]),
                             'Confidence':'HIGH' if residual[a,b]<=10 else 'REVIEW'})
COAL_MATCHES=pd.DataFrame(coal_matches,columns=['Regional_Name','Well',
    'Top_TWT_ms','Base_TWT_ms','Thickness_m','Residual_m','Confidence'])
COAL_MATCHES.to_csv(OUTPUT_ROOT/'Phase5_Proposed_Regional_Coal_Correlations.csv',index=False)
display(COAL_MATCHES.head(30))
if ref_coals.empty:
    print('Coal matching skipped: no reference thick coal; marker correlation remains active.')

print("Cell 7 completed")


I-50_COAL: thickness exemption applied. Existing marker TWT is retained as the regional anchor; no coal-bed top/base is fabricated.


,Regional_Name,Well,Top_TWT_ms,Base_TWT_ms,Thickness_m,Residual_m,Confidence
0,REG_COAL_01,Well_4,1435.361077,1436.473356,1.523991,0.000000,REFERENCE
1,REG_COAL_02,Well_4,1717.751219,1718.790581,1.523991,0.000000,REFERENCE
2,REG_COAL_03,Well_4,2047.921939,2048.786300,1.523909,0.000000,REFERENCE
3,REG_COAL_01,Well_1,1448.714195,1451.178195,3.352800,7.479016,HIGH
4,REG_COAL_02,Well_1,1741.877047,1743.032047,1.676400,17.633685,REVIEW
5,REG_COAL_01,Well_2,1485.419077,1493.620734,11.733415,19.919468,REVIEW
6,REG_COAL_02,Well_5,1742.059086,1744.031951,3.045888,5.666493,HIGH
7,REG_COAL_02,Well_7,1744.412200,1745.344363,1.524000,6.069218,HIGH


Cell 7 completed


In [8]:
# CELL 8 — Phase 5 TWT well correlation
# Construct a monotonic piecewise depth mapping: target well TVDSS -> reference TVDSS.
# Measured marker pairs and accepted thick-coal matches provide control points.
# Reject any pair creating unreasonable local stratigraphic stretch/squeeze.
WARPS={}; WARP_QC=[]
ref_marks=MEASURED[MEASURED.Well==REFERENCE_WELL].groupby('Canonical').TWT_ms.median()
ref_coal_mid=ref_coals.set_index('Regional_Name').Mid_TWT_ms
for well in WELL_DATA:
    if well==REFERENCE_WELL:
        WARPS[well]=(np.array([-1e6,1e6]),np.array([-1e6,1e6]))
        continue
    if not np.isfinite(REGIONAL_OFFSETS[well]):
        WARP_QC.append({'Well':well,'Status':'UNANCHORED','Control_points':0})
        continue
    pts=[]
    local=MEASURED[MEASURED.Well==well].groupby('Canonical').TWT_ms.median()
    for name,z in local.items():
        if name in ref_marks.index:
            pts.append((float(z),float(ref_marks[name]),'MEASURED_MARKER',name))
    for _,r in COAL_MATCHES[COAL_MATCHES.Well==well].iterrows():
        if r.Regional_Name in ref_coal_mid:
            pts.append(((r.Top_TWT_ms+r.Base_TWT_ms)/2,
                        float(ref_coal_mid[r.Regional_Name]),'THICK_COAL',r.Regional_Name))
    pts.sort()
    # Select controls in increasing order, favoring measured markers on ties.
    accepted=[]
    for z,r,kind,name in pts:
        if accepted and z-accepted[-1][0]<0.5:continue
        if accepted:
            slope=(r-accepted[-1][1])/(z-accepted[-1][0])
            if not MIN_WARP_LOCAL_SLOPE<=slope<=MAX_WARP_LOCAL_SLOPE:
                continue
        accepted.append((z,r,kind,name))
    if not accepted:
        WARP_QC.append({'Well':well,'Status':'NO_VALID_CONTROLS','Control_points':0})
        continue
    x=np.array([p[0] for p in accepted])
    y=np.array([p[1] for p in accepted])
    if len(x)==1:
        x=np.r_[x[0]-1000,x[0],x[0]+1000]
        y=np.r_[y[0]-1000,y[0],y[0]+1000]
    WARPS[well]=(x,y)
    WARP_QC.append({'Well':well,'Status':'READY','Control_points':len(accepted),
                    'Measured_controls':sum(p[2]=='MEASURED_MARKER' for p in accepted),
                    'Coal_controls':sum(p[2]=='THICK_COAL' for p in accepted)})
WARP_QC=pd.DataFrame(WARP_QC)
WARP_QC.to_csv(OUTPUT_ROOT/'Phase5_Warp_QC.csv',index=False)
display(WARP_QC)

def to_reference_depth(well,z):
    z=np.asarray(z,dtype=float)
    if well not in WARPS:return np.full_like(z,np.nan)
    x,y=WARPS[well]
    mapped=np.interp(z,x,y)
    left=z<x[0];right=z>x[-1]
    mapped[left]=y[0]+(z[left]-x[0])*(y[1]-y[0])/(x[1]-x[0])
    mapped[right]=y[-1]+(z[right]-x[-1])*(y[-1]-y[-2])/(x[-1]-x[-2])
    far=(z<x[0]-MAX_WARP_EXTRAPOLATION_MS)|(z>x[-1]+MAX_WARP_EXTRAPOLATION_MS)
    # A one-control affine map has no real depth coverage: don't mask it here.
    if len(x)>3:mapped[far]=np.nan
    return mapped

def from_reference_depth(well,z):
    z=np.asarray(z,dtype=float)
    if well not in WARPS:return np.full_like(z,np.nan)
    x,y=WARPS[well]
    mapped=np.interp(z,y,x)
    left=z<y[0];right=z>y[-1]
    mapped[left]=x[0]+(z[left]-y[0])*(x[1]-x[0])/(y[1]-y[0])
    mapped[right]=x[-1]+(z[right]-y[-1])*(x[-1]-x[-2])/(y[-1]-y[-2])
    if len(x)>3:
        mapped[(z<y[0]-MAX_WARP_EXTRAPOLATION_MS)|
               (z>y[-1]+MAX_WARP_EXTRAPOLATION_MS)]=np.nan
    return mapped

print("Cell 8 completed")


,Well,Status,Control_points,Measured_controls,Coal_controls
0,Well_1,READY,3,1,2
1,Well_2,READY,8,7,1
2,Well_5,READY,10,9,1
3,Well_6,READY,10,10,0
4,Well_7,READY,7,6,1
5,Well_8,READY,4,4,0


Cell 8 completed


In [9]:
# CELL 9 — Phase 5 TWT well correlation
# Propose missing marker positions and conservative measured-marker adjustments.
# Nearest local log signature is sought within the configured TWT (ms) tolerance
# around the warp-projected time. Signal correlation is advisory, not proof.
def feature_signature(df):
    cols=[c for c in ('GR_P','DENS_P','RHOBHC','DTC_P','FCOAL','VCLB') if c in df]
    if not cols:return None
    z=df.TWT_ms.to_numpy(float)
    feats=[]
    for c in cols:
        v=df[c].to_numpy(float)
        if np.isfinite(v).sum()<20:continue
        med=np.nanmedian(v);mad=np.nanmedian(np.abs(v-med))
        v=(v-med)/max(1.4826*mad,1e-6)
        v=np.interp(z,z[np.isfinite(v)],v[np.isfinite(v)])
        feats.append(np.clip(v,-4,4))
    return np.stack(feats,axis=1) if feats else None

SIGNATURES={w:feature_signature(d) for w,d in WELL_DATA.items()}
def log_match_score(well,depth,ref_depth,half_window=8.0,return_detail=False):
    """Weighted multilog NCC; GR dominates. Correlate only shared valid curves.

    Weights are renormalized over usable curves; report curve-level scores.
    Does not treat lithology predictions as independent measured evidence.
    """
    da,db=WELL_DATA[well],WELL_DATA[REFERENCE_WELL]
    offsets=np.arange(-half_window,half_window+0.5,1.0)
    components={}; usable_weights={}
    for col,weight in LOG_CORRELATION_WEIGHTS.items():
        if col not in da or col not in db:continue
        za,zb=da.TWT_ms.to_numpy(float),db.TWT_ms.to_numpy(float)
        va,vb=da[col].to_numpy(float),db[col].to_numpy(float)
        oka=np.isfinite(va)&np.isfinite(za)
        okb=np.isfinite(vb)&np.isfinite(zb)
        if oka.sum()<10 or okb.sum()<10:continue
        ia=np.argsort(za[oka]);ib=np.argsort(zb[okb])
        xa,xb=za[oka][ia],zb[okb][ib]
        ya,yb=va[oka][ia],vb[okb][ib]
        if depth+offsets[0]<xa[0] or depth+offsets[-1]>xa[-1]:continue
        if ref_depth+offsets[0]<xb[0] or ref_depth+offsets[-1]>xb[-1]:continue
        x=np.interp(depth+offsets,xa,ya)
        y=np.interp(ref_depth+offsets,xb,yb)
        if np.std(x)<1e-8 or np.std(y)<1e-8:continue
        corr=float(np.corrcoef(x,y)[0,1])
        if not np.isfinite(corr):continue
        components[col]=corr
        usable_weights[col]=weight
    if GR_PRIMARY_REQUIRED and 'GR_LAS' not in components:
        components={};usable_weights={}
    denom=sum(usable_weights.values())
    score=(sum(components[k]*usable_weights[k] for k in components)/denom
           if denom>0 else np.nan)
    if return_detail:return score,components
    return score

proposals=[]
DIAG={'Marker_well_pairs':0,'Missing_warp':0,'Nonfinite_warp_target':0,
      'No_valid_TWT_samples':0,'Outside_log_TWT_range':0,'Proposals_created':0}
# Reference depths for every approved marker, using all available measured ties
# transformed to reference stratigraphy. Require at least one valid mapped tie.
regional_marker_depths={}
for name in MATRIX.Marker_Name:
    ties=MEASURED[MEASURED.Canonical==name]
    mapped=[]
    for _,tie in ties.iterrows():
        if tie.Well in WARPS:
            rz=to_reference_depth(tie.Well,np.array([tie.TWT_ms]))[0]
            if np.isfinite(rz):mapped.append(float(rz))
    if mapped:regional_marker_depths[name]=float(np.median(mapped))
print('Approved markers with usable regional TWT ties:',
      len(regional_marker_depths),'of',len(MATRIX))
for name,ref_depth in regional_marker_depths.items():
    if name in EXCLUDE_MARKERS:continue
    for well,df in WELL_DATA.items():
        if well==REFERENCE_WELL:continue
        DIAG['Marker_well_pairs']+=1
        if well not in WARPS:
            DIAG['Missing_warp']+=1
            continue
        target=float(from_reference_depth(well,np.array([ref_depth]))[0])
        if not np.isfinite(target):
            DIAG['Nonfinite_warp_target']+=1
            continue
        measured=MEASURED[(MEASURED.Well==well)&(MEASURED.Canonical==name)]
        exists=not measured.empty
        original=float(measured.TWT_ms.median()) if exists else np.nan
        center=original if exists else target
        maxshift=MAX_MARKER_ADJUST_MS if exists else MAX_MARKER_PROPOSE_DISTANCE_MS
        depths=df.TWT_ms.to_numpy(float)
        valid_times=depths[np.isfinite(depths)]
        if valid_times.size<10:
            DIAG['No_valid_TWT_samples']+=1
            continue
        lo,hi=float(valid_times.min()),float(valid_times.max())
        if not (lo<=center<=hi):
            DIAG['Outside_log_TWT_range']+=1
            continue
        # Select small local candidate shifts by multi-log waveform similarity.
        candidates=np.linspace(max(lo,center-maxshift),
                               min(hi,center+maxshift),25)
        scores=(np.array([log_match_score(well,float(z),float(ref_depth)) for z in candidates])
                if name in ref_marks.index else np.full(len(candidates),np.nan))
        residual_penalty=np.abs(candidates-target)/max(maxshift,1.)
        objective=np.where(np.isfinite(scores),scores,0.)-0.22*residual_penalty
        j=int(np.argmax(objective))
        best=float(candidates[j]);score=float(scores[j]) if np.isfinite(scores[j]) else np.nan
        shift=best-original if exists else np.nan
        confidence=('REVIEW' if np.isfinite(score) and score>=MIN_CORRELATION_SCORE
                    else 'LOW_CONFIDENCE')
        if exists and abs(shift)<1.:status='KEEP_EXISTING'
        elif exists:status='ADJUSTMENT_PROPOSED'
        else:status='MISSING_MARKER_PROPOSED'
        proposals.append({'Well':well,'Canonical':name,'Status':status,
                          'Original_TWT_ms':original,'Warp_Target_TWT_ms':target,
                          'Proposed_TWT_ms':best,'Suggested_Shift_ms':shift,
                          'Log_Correlation':score,'GR_Correlation':(log_match_score(well,best,ref_depth,return_detail=True)[1].get('GR_LAS',np.nan) if name in ref_marks.index else np.nan),
                          'Curves_Used':('|'.join(log_match_score(well,best,ref_depth,return_detail=True)[1]) if name in ref_marks.index else ''),
                          'Confidence':confidence,
                          'Anchor_Method':ANCHOR_METHOD[well],
                          'Requires_Interpreter_Approval':True})
        DIAG['Proposals_created']+=1
PROPOSAL_COLUMNS=['Well','Canonical','Status','Original_TWT_ms',
                  'Warp_Target_TWT_ms','Proposed_TWT_ms','Suggested_Shift_ms',
                  'Log_Correlation','GR_Correlation','Curves_Used','Confidence','Anchor_Method',
                  'Requires_Interpreter_Approval']
PROPOSALS=pd.DataFrame(proposals,columns=PROPOSAL_COLUMNS)

# ---------------------------------------------------------------------------
# STRATIGRAPHIC CROSSOVER GUARD: performed BEFORE exporting proposals.
# All measured markers are immutable. Use interpreter-specified stratigraphic
# order, NOT reference Well 4 TWT. Unlisted markers remain unranked/review.
# Proposal acceptance is deterministic, confidence-first and non-destructive.
# ---------------------------------------------------------------------------
REF_MARKER_ORDER=list(STRATIGRAPHIC_ORDER)
REF_MARKER_RANK=dict(STRATIGRAPHIC_RANK)
print('Using interpreter-defined stratigraphic order:',len(REF_MARKER_ORDER),'markers')
CROSSOVER_EVENTS=[]

def _record_crossover(well,kind,shallower,deeper,shallower_twt,deeper_twt,proposal='',detail=''):
    CROSSOVER_EVENTS.append({'Well':well,'Issue':kind,'Shallower':shallower,
        'Deeper':deeper,'Shallower_TWT_ms':shallower_twt,'Deeper_TWT_ms':deeper_twt,
        'Proposal':proposal,'Detail':detail})

def _violations(name,twt,placed):
    """Return conflicts with all placed controls, not just adjacent proposals."""
    r=REF_MARKER_RANK[name]
    out=[]
    for other,(other_twt,other_type) in placed.items():
        if other==name or other not in REF_MARKER_RANK:continue
        ro=REF_MARKER_RANK[other]
        if ro<r and twt-other_twt<MIN_MARKER_SEPARATION_MS-1e-7:
            out.append((other,name,other_twt,twt,other_type))
        elif ro>r and other_twt-twt<MIN_MARKER_SEPARATION_MS-1e-7:
            out.append((name,other,twt,other_twt,other_type))
    return out

for column,default in [('QC_Decision','UNREVIEWED'),('QC_Reason',''),
                       ('Effective_TWT_ms',np.nan),('Conflicting_With','')]:
    PROPOSALS[column]=default

for well in WELL_NAMES:
    measured=MEASURED[(MEASURED.Well==well)&MEASURED.Canonical.isin(REF_MARKER_RANK)].copy()
    measured['TWT_ms']=pd.to_numeric(measured['TWT_ms'],errors='coerce')
    measured=measured[np.isfinite(measured['TWT_ms'])]
    measured_times=measured.groupby('Canonical')['TWT_ms'].median().to_dict()
    # Every measured marker is an immutable hard control, even if it conflicts
    # with another measured marker. Such conflicts require interpreter review.
    fixed={name:(float(twt),'MEASURED') for name,twt in measured_times.items()}
    sorted_measured=sorted(fixed,key=lambda name:REF_MARKER_RANK[name])
    for left,right in zip(sorted_measured[:-1],sorted_measured[1:]):
        t1,t2=fixed[left][0],fixed[right][0]
        if t2-t1<MIN_MARKER_SEPARATION_MS-1e-7:
            _record_crossover(well,'MEASURED_MARKER_ORDER_CONFLICT',left,right,t1,t2,
                              detail='Measured markers preserved unchanged; interpreter review required')
    # I-50_COAL is already protected because it is a measured hard control.
    # If not measured, no I-50_COAL position is invented or force-anchored.
    placed=dict(fixed)
    rows=PROPOSALS.index[PROPOSALS.Well==well].tolist()
    # Evaluate existing-marker adjustments first (measured TWT stays fixed),
    # then missing markers by evidence strength, with deterministic tie-breaks.
    def _priority(idx):
        p=PROPOSALS.loc[idx]
        existing=np.isfinite(p.Original_TWT_ms)
        confidence=0 if p.Confidence=='REVIEW' else 1
        score=-float(p.Log_Correlation) if np.isfinite(p.Log_Correlation) else 1.0
        return (0 if existing else 1,confidence,score,
                REF_MARKER_RANK.get(p.Canonical,10**9),str(p.Canonical))
    for idx in sorted(rows,key=_priority):
        p=PROPOSALS.loc[idx]; name=p.Canonical
        original=float(p.Original_TWT_ms) if np.isfinite(p.Original_TWT_ms) else np.nan
        proposed=float(p.Proposed_TWT_ms) if np.isfinite(p.Proposed_TWT_ms) else np.nan
        if name not in REF_MARKER_RANK:
            PROPOSALS.at[idx,'QC_Decision']='REVIEW_NO_REFERENCE_ORDER'
            PROPOSALS.at[idx,'QC_Reason']='Marker absent from interpreter-defined stratigraphic order'
            if np.isfinite(original):PROPOSALS.at[idx,'Effective_TWT_ms']=original
            continue
        if np.isfinite(original):
            # A proposed adjustment is never accepted over an existing measured
            # tie; it is only a recommendation requiring manual approval.
            conflicts=_violations(name,proposed,{k:v for k,v in placed.items() if k!=name}) if np.isfinite(proposed) else []
            if conflicts:
                PROPOSALS.at[idx,'QC_Decision']='REJECTED_CROSSOVER'
                PROPOSALS.at[idx,'QC_Reason']='Proposed adjustment crosses protected measured/accepted marker'
                PROPOSALS.at[idx,'Conflicting_With']='|'.join(sorted(set(
                    b if a==name else a for a,b,*_ in conflicts)))
                for a,b,ta,tb,kind in conflicts:
                    _record_crossover(well,'PROPOSED_MARKER_CROSSOVER',a,b,ta,tb,name,
                                      'Adjustment rejected; original measured TWT retained')
            else:
                PROPOSALS.at[idx,'QC_Decision']='KEEP_MEASURED' if p.Status=='KEEP_EXISTING' else 'ADJUSTMENT_FOR_REVIEW'
                PROPOSALS.at[idx,'QC_Reason']='Measured marker remains unchanged until interpreter approval'
            PROPOSALS.at[idx,'Effective_TWT_ms']=original
            continue
        if not np.isfinite(proposed):
            PROPOSALS.at[idx,'QC_Decision']='REJECTED_INVALID_TWT'
            PROPOSALS.at[idx,'QC_Reason']='Nonfinite proposed TWT'
            continue
        conflicts=_violations(name,proposed,placed)
        if conflicts:
            PROPOSALS.at[idx,'QC_Decision']='REJECTED_CROSSOVER'
            PROPOSALS.at[idx,'QC_Reason']='Conflicts with measured or higher-priority provisional marker'
            PROPOSALS.at[idx,'Conflicting_With']='|'.join(sorted(set(
                b if a==name else a for a,b,*_ in conflicts)))
            for a,b,ta,tb,kind in conflicts:
                _record_crossover(well,'PROPOSED_MARKER_CROSSOVER',a,b,ta,tb,name,
                                  'Missing-marker proposal rejected; no forced placement')
        else:
            # This is provisional only: not an approved marker or measured tie.
            placed[name]=(proposed,'PROVISIONAL')
            PROPOSALS.at[idx,'QC_Decision']='PROVISIONAL_NONCROSSING'
            PROPOSALS.at[idx,'QC_Reason']='No crossover; interpreter approval still required'
            PROPOSALS.at[idx,'Effective_TWT_ms']=proposed

CROSSOVER_EVENTS=pd.DataFrame(CROSSOVER_EVENTS,columns=[
    'Well','Issue','Shallower','Deeper','Shallower_TWT_ms','Deeper_TWT_ms',
    'Proposal','Detail'])
CROSSOVER_EVENTS.to_csv(OUTPUT_ROOT/'Phase5_Crossover_Decision_Audit.csv',index=False)
PROPOSALS['Requires_Interpreter_Approval']=True
print('Crossover decisions:')
if not PROPOSALS.empty:
    display(PROPOSALS.groupby('QC_Decision').size().rename('Count').reset_index())
print('Crossover audit events:',len(CROSSOVER_EVENTS))

print('Cell 9 diagnostics:')
display(pd.DataFrame([DIAG]))
print('Wells with TWT logs:',list(WELL_DATA))
print('Wells with warps:',list(WARPS))
PROPOSALS.to_csv(OUTPUT_ROOT/'Phase5_Marker_Correlation_Proposals.csv',index=False)
# All measured markers remain untouched in a separate output.
MEASURED.to_csv(OUTPUT_ROOT/'Phase5_Original_Approved_Markers_Unchanged.csv',index=False)
if PROPOSALS.empty:
    print('WARNING: No correlation proposals created. Inspect the diagnostics above.')
    print('Typical causes: missing warp controls, insufficient time-depth coverage,')
    print('or proposed markers falling outside available TWT log ranges.')
else:
    display(PROPOSALS.groupby(['Status','Confidence']).size().rename('Count').reset_index())

print("Cell 9 completed")


Approved markers with usable regional TWT ties: 35 of 45
Using interpreter-defined stratigraphic order: 46 markers
Crossover decisions:


,QC_Decision,Count
0,ADJUSTMENT_FOR_REVIEW,33
1,KEEP_MEASURED,31
2,PROVISIONAL_NONCROSSING,74
3,REJECTED_CROSSOVER,18


Crossover audit events: 19
Cell 9 diagnostics:


,Marker_well_pairs,Missing_warp,Nonfinite_warp_target,No_valid_TWT_samples,Outside_log_TWT_range,Proposals_created
0,210,0,45,0,9,156


Wells with TWT logs: ['Well_1', 'Well_2', 'Well_4', 'Well_5', 'Well_6', 'Well_7', 'Well_8']
Wells with warps: ['Well_1', 'Well_2', 'Well_4', 'Well_5', 'Well_6', 'Well_7', 'Well_8']


,Status,Confidence,Count
0,ADJUSTMENT_PROPOSED,LOW_CONFIDENCE,25
1,ADJUSTMENT_PROPOSED,REVIEW,8
2,KEEP_EXISTING,LOW_CONFIDENCE,24
3,KEEP_EXISTING,REVIEW,7
4,MISSING_MARKER_PROPOSED,LOW_CONFIDENCE,81
5,MISSING_MARKER_PROPOSED,REVIEW,11


Cell 9 completed


In [10]:
# CELL 10 — Independent stratigraphic QC after Cell 9 crossover decisions
# Check effective positions (original measured ties + noncrossing provisional picks).
# Do not auto-move or delete existing measured markers.
if 'REF_MARKER_RANK' not in globals():
    raise RuntimeError('Run Cell 9 first to generate crossover decisions.')
order_issues=[]
for well in WELL_NAMES:
    measured=MEASURED[(MEASURED.Well==well)&MEASURED.Canonical.isin(REF_MARKER_RANK)].copy()
    measured['TWT_ms']=pd.to_numeric(measured['TWT_ms'],errors='coerce')
    measured=measured[np.isfinite(measured['TWT_ms'])]
    positions={k:float(v) for k,v in measured.groupby('Canonical')['TWT_ms'].median().items()}
    for _,p in PROPOSALS[(PROPOSALS.Well==well)&
                          (PROPOSALS.QC_Decision=='PROVISIONAL_NONCROSSING')].iterrows():
        if p.Canonical not in positions and np.isfinite(p.Effective_TWT_ms):
            positions[p.Canonical]=float(p.Effective_TWT_ms)
    ordered=sorted(positions,key=lambda name:REF_MARKER_RANK[name])
    for left,right in zip(ordered[:-1],ordered[1:]):
        gap=positions[right]-positions[left]
        if gap<MIN_MARKER_SEPARATION_MS-1e-7:
            order_issues.append({'Well':well,'Shallower':left,'Deeper':right,
                'Shallower_TWT_ms':positions[left], 'Deeper_TWT_ms':positions[right],
                'Gap_ms':gap,'Issue':'MEASURED_OR_EFFECTIVE_ORDER_CONFLICT',
                'Action':'INTERPRETER_REVIEW_NO_AUTOMATIC_CORRECTION'})
ORDER_QC=pd.DataFrame(order_issues,columns=[
    'Well','Shallower','Deeper','Shallower_TWT_ms','Deeper_TWT_ms','Gap_ms','Issue','Action'])
ORDER_QC.to_csv(OUTPUT_ROOT/'Phase5_Stratigraphic_Order_Flags.csv',index=False)
print('Independent residual stratigraphic QC issues:',len(ORDER_QC))
if not ORDER_QC.empty:display(ORDER_QC.head(30))
else:print('PASS: No remaining crossover in effective marker positions.')
print('Rejected proposal count:',int((PROPOSALS.QC_Decision=='REJECTED_CROSSOVER').sum()))
print('Original measured markers were not modified.')
print('Cell 10 completed')


Independent residual stratigraphic QC issues: 0
PASS: No remaining crossover in effective marker positions.
Rejected proposal count: 18
Original measured markers were not modified.
Cell 10 completed


In [11]:
# CELL 11 — Interactive TWT zoom/pan, GR line only, large marker labels
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import patheffects

# Resolve actual GR mnemonics, rather than silently requiring GR_P.
GR_CANDIDATES=('GR_LAS','GR_P','GR','GR_N','GR_API','GAMMA_RAY','GAMMARAY',
               'GRC','GR_CORR','CGR','SGR','GR_RAW','GR_LOG')
def resolve_gr_curve(df):
    by_name={re.sub(r'[^A-Z0-9]','',str(c).upper()):c for c in df.columns}
    for name in GR_CANDIDATES:
        key=re.sub(r'[^A-Z0-9]','',name.upper())
        if key in by_name:
            col=by_name[key]
            vals=pd.to_numeric(df[col],errors='coerce').to_numpy(float)
            if np.isfinite(vals).sum()>=10:
                return col,vals
    return None,None

GR_PANEL_STATUS=[]
for _well,_df in WELL_DATA.items():
    _name,_values=resolve_gr_curve(_df)
    _times=pd.to_numeric(_df['TWT_ms'],errors='coerce').to_numpy(float)
    _count=int(np.sum(np.isfinite(_times)&np.isfinite(_values))) if _values is not None else 0
    GR_PANEL_STATUS.append({'Well':_well,'GR_column':_name or 'NOT FOUND',
                            'Valid_GR_TWT_samples':_count})
GR_PANEL_STATUS=pd.DataFrame(GR_PANEL_STATUS)
print('GR source diagnostics (GR_LAS comes from the original LAS, aligned via MDKB_m):')
display(GR_PANEL_STATUS)
GR_PANEL_STATUS.to_csv(OUTPUT_ROOT/'Phase5_GR_Source_Diagnostics.csv',index=False)

well_picker=widgets.SelectMultiple(options=list(WELL_DATA),value=tuple(WELL_DATA),
                                   description='Wells',rows=min(7,len(WELL_DATA)),
                                   layout=widgets.Layout(width='210px'))
show_proposals=widgets.Checkbox(value=True,description='Proposed markers')
show_coals=widgets.Checkbox(value=True,description=f'Coal >{MIN_COAL_THICKNESS_M:g} m')
show_labels=widgets.Checkbox(value=True,description='Marker labels')
view_mode=widgets.Dropdown(options=[('Aligned TWT (Well 4 datum)','aligned'),
                                    ('Original TWT (ms)','original')],value='aligned',description='Domain')
font_control=widgets.IntSlider(value=max(14,PANEL_FONT_SIZE),min=11,max=24,step=1,description='Font')
marker_control=widgets.IntSlider(value=max(14,PANEL_MARKER_FONT_SIZE),min=10,max=24,step=1,description='Labels')
# TWT navigator: zoom by changing the interval width; pan with the range slider
# or with the up/down buttons. All values are in the selected display domain.
def _get_twt_extent(mode):
    chunks=[]
    for _well,_df in WELL_DATA.items():
        _t=pd.to_numeric(_df['TWT_ms'],errors='coerce').to_numpy(float)
        _t=_panel_time(_well,_t,mode)
        _t=_t[np.isfinite(_t)]
        if len(_t):chunks.append(_t)
    if not chunks:return 0.,3000.
    vals=np.concatenate(chunks)
    return float(np.floor(np.nanmin(vals)/10)*10),float(np.ceil(np.nanmax(vals)/10)*10)

# _panel_time is defined below; extent initialization happens after its definition.
plot_output=widgets.Output()

def _panel_time(well,values,mode):
    v=np.asarray(values,dtype=float)
    return to_reference_depth(well,v) if mode=='aligned' else v


# Initialize navigator after _panel_time exists.
_twt_min,_twt_max=_get_twt_extent(view_mode.value)
_twt_step=1.0
if _twt_max-_twt_min<2:_twt_max=_twt_min+2
twt_window=widgets.FloatRangeSlider(value=(_twt_min,_twt_max),min=_twt_min,max=_twt_max,
                                    step=_twt_step,description='TWT (ms)',continuous_update=False,
                                    readout_format='.0f',layout=widgets.Layout(width='700px'),
                                    style={'description_width':'initial'})
pan_up=widgets.Button(description='Pan shallower',icon='arrow-up')
pan_down=widgets.Button(description='Pan deeper',icon='arrow-down')
reset_twt=widgets.Button(description='Full TWT',icon='maximize')
def _pan_twt(direction):
    lo,hi=twt_window.value
    shift=direction*max(1.,(hi-lo)*.25)
    shift=max(twt_window.min-lo,min(shift,twt_window.max-hi))
    twt_window.value=(lo+shift,hi+shift)
def _reset_twt(_=None):
    twt_window.value=(twt_window.min,twt_window.max)
def _on_domain_change(change):
    if change['name']!='value':return
    lo,hi=_get_twt_extent(change['new'])
    if hi-lo<2:hi=lo+2
    twt_window.min=min(twt_window.min,lo)
    twt_window.max=max(twt_window.max,hi)
    twt_window.min=lo;twt_window.max=hi
    twt_window.value=(lo,hi)
pan_up.on_click(lambda _: _pan_twt(-1))
pan_down.on_click(lambda _: _pan_twt(+1))
reset_twt.on_click(_reset_twt)
view_mode.observe(_on_domain_change,names='value')

def draw_panel(_=None,save=False):
    selected=list(well_picker.value)
    if not selected:return
    fs=font_control.value
    with plot_output:
        plot_output.clear_output(wait=True)
        fig,axes=plt.subplots(1,len(selected),figsize=(max(15,3.3*len(selected)),14),
                              sharey=True,squeeze=False)
        axes=axes[0]
        gr_max=180.0
        for ax,well in zip(axes,selected):
            df=WELL_DATA[well]
            t=df.TWT_ms.to_numpy(float)
            y=_panel_time(well,t,view_mode.value)
            gr_name,gr=resolve_gr_curve(df)
            if gr is not None:
                valid=np.isfinite(gr)&np.isfinite(y)
                if np.count_nonzero(valid)>=10:
                    # Sort by plotted time; discontinuous NaN samples must not
                    # create artificial lines through missing intervals.
                    xx=np.clip(gr[valid],0,gr_max)
                    yy=y[valid]
                    order=np.argsort(yy)
                    xx,yy=xx[order],yy[order]
                    ax.plot(xx,yy,color='#182c3a',lw=1.8,zorder=3,
                            label=f'GR ({gr_name})')
                else:
                    ax.text(.5,.5,f'GR unavailable\n{gr_name}: <10 valid TWT samples',
                            transform=ax.transAxes,ha='center',va='center',
                            fontsize=fs-1,color='#b91c1c')
            else:
                ax.text(.5,.5,'GR column not found\nCheck GR source diagnostics',
                        transform=ax.transAxes,ha='center',va='center',
                        fontsize=fs-1,color='#b91c1c')
            ax.set_xlim(0,gr_max)
            ax.set_facecolor('#ffffff')
            ax.xaxis.set_ticks([0,50,100,150])
            ax.tick_params(axis='both',labelsize=fs-1)
            ax.grid(axis='y',color='#cbd5e1',alpha=.6,lw=.65,zorder=0)
            ax.set_axisbelow(True)
            if show_coals.value and not COAL_MATCHES.empty:
                for _,c in COAL_MATCHES[COAL_MATCHES.Well==well].iterrows():
                    bounds=_panel_time(well,[c.Top_TWT_ms,c.Base_TWT_ms],view_mode.value)
                    if np.all(np.isfinite(bounds)):
                        ax.axhspan(min(bounds),max(bounds),facecolor='#475569',alpha=.25,zorder=2)
            measured=MEASURED[MEASURED.Well==well]
            for _,m in measured.iterrows():
                yt=_panel_time(well,[m.TWT_ms],view_mode.value)[0]
                if not np.isfinite(yt):continue
                is_i50=(m.Canonical==REGIONAL_COAL_ANCHOR)
                col='#7c3aed' if is_i50 else '#0b7285'
                ax.axhline(yt,color=col,lw=1.8 if is_i50 else 1.05,zorder=5)
                if show_labels.value:
                    label=ax.text(4,yt,m.Canonical,fontsize=marker_control.value,
                                  color=col,va='bottom',weight='bold' if is_i50 else 'medium',
                                  clip_on=True,zorder=7)
                    label.set_path_effects([patheffects.withStroke(linewidth=2.5,foreground='white')])
            if show_proposals.value and not PROPOSALS.empty:
                proposals=PROPOSALS[(PROPOSALS.Well==well)&
                                    (PROPOSALS.QC_Decision.isin(['PROVISIONAL_NONCROSSING','ADJUSTMENT_FOR_REVIEW']))]
                for _,p in proposals.iterrows():
                    yt=_panel_time(well,[p.Proposed_TWT_ms],view_mode.value)[0]
                    if not np.isfinite(yt):continue
                    col='#b45309' if p.Confidence=='REVIEW' else '#c2410c'
                    ax.axhline(yt,color=col,lw=.9,ls='--',alpha=.65,zorder=4)
            ax.set_title(well.replace('_',' '),fontsize=fs+3,weight='bold',pad=15)
            ax.set_xlabel(f'Gamma Ray ({gr_name or "missing"})',fontsize=fs,weight='medium',labelpad=9)
            for side in ('top','right'):ax.spines[side].set_visible(False)
        # Explicit view limits, inverted for seismic TWT (younger at top).
        _top,_base=twt_window.value
        for _ax in axes:_ax.set_ylim(_base,_top)
        axes[0].set_ylabel(('Aligned TWT (ms) — Well 4 reference' if view_mode.value=='aligned'
                            else 'Original TWT (ms)'),fontsize=fs+2,weight='bold')
        fig.suptitle('REGIONAL WELL CORRELATION | SEVEN-WELL PANEL',
                     fontsize=fs+7,weight='bold',y=.992)
        fig.text(.5,.952,'GR-primary multilog correlation  •  I-50_COAL regional anchor  •  Proposals require geological QC',
                 ha='center',fontsize=fs,color='#475569')
        handles=[Line2D([0],[0],color='#263746',lw=1.5,label='Gamma ray'),
                 Line2D([0],[0],color='#0b7285',lw=1.5,label='Existing marker'),
                 Line2D([0],[0],color='#7c3aed',lw=2,label='I-50_COAL'),
                 Line2D([0],[0],color='#b45309',ls='--',lw=1.5,label='Proposed marker'),
                 Patch(facecolor='#475569',alpha=.25,label=f'Coal >{MIN_COAL_THICKNESS_M:g} m')]
        fig.legend(handles=handles,loc='lower center',ncol=5,fontsize=fs-1,
                   frameon=False,bbox_to_anchor=(.5,.015))
        fig.subplots_adjust(top=.90,bottom=.12,left=.065,right=.985,wspace=.08)
        if save:
            stem='Phase5_Seven_Well_Correlation_'+view_mode.value
            png=OUTPUT_ROOT/(stem+'.png');pdf=OUTPUT_ROOT/(stem+'.pdf')
            fig.savefig(png,dpi=PANEL_DPI,bbox_inches='tight',facecolor='white')
            fig.savefig(pdf,bbox_inches='tight',facecolor='white')
            print('Saved:',png,'and',pdf)
        display(fig)
        plt.close(fig)

for control in (well_picker,show_proposals,show_coals,show_labels,
                view_mode,font_control,marker_control,twt_window):
    control.observe(draw_panel,names='value')
save_button=widgets.Button(description='Export PNG + PDF',button_style='success',icon='download')
save_button.on_click(lambda _:draw_panel(save=True))
display(widgets.VBox([
    widgets.HTML('<b>TWT NAVIGATOR</b> — drag either handle to zoom; drag the selected interval to pan. Up/down buttons shift by 25% of the current window.'),
    widgets.HBox([twt_window,pan_up,pan_down,reset_twt]),
    widgets.HBox([well_picker,widgets.VBox([view_mode,show_proposals,show_coals,
                                          show_labels,font_control,marker_control,
                                          save_button])]),plot_output]))
draw_panel()
print('Cell 11 completed')


GR source diagnostics (GR_LAS comes from the original LAS, aligned via MDKB_m):


,Well,GR_column,Valid_GR_TWT_samples
0,Well_1,GR_LAS,17910
1,Well_2,GR_LAS,19393
2,Well_4,GR_LAS,9824
3,Well_5,GR_LAS,7145
4,Well_6,GR_LAS,7763
5,Well_7,GR_LAS,6927
6,Well_8,GR_LAS,6547


Cell 11 completed


## QC and interpretation notes

1. **I-50_COAL is the primary regional datum**, but the supplied Step 5 matrix has this marker only in Wells 4 and 7. Other wells therefore need provisional shared-marker anchoring and must be reviewed.
2. **Thick coal beds** are candidate regional markers only when supported by the available lithology/fraction data and stratigraphic alignment. `REG_COAL_01`, etc. are provisional names.
3. **Well 3 is excluded completely** because it is outside seismic coverage.
4. **Small shifts** are proposed, never applied to the original markers. Inspect `Phase5_Marker_Correlation_Proposals.csv`, particularly low-confidence scores, local stretch/squeeze, and crossing flags.
5. **Important limitation:** training-well Phase 1–3 CSV filenames were not available for verification in this environment. Set `WELL_RESULT_OVERRIDES` to the actual exports if source status shows missing wells. A complete all-eight-well run requires all seven included input result files.
6. This notebook uses constrained signal alignment and robust assignment; it does not train a supervised ML model without labelled interwell correlations. A later supervised model can be fitted after manual approval of these proposals.

**Domain requirement:** All correlation, log alignment, warp and QC plots use **TWT (ms)**. Coal thickness is measured separately in **TVDSS (m)**. Missing absolute time-depth ties cause a stop, not a fabricated time scale.

**Editable threshold:** `MIN_COAL_THICKNESS_M = 1.5` in Cell 1. The minimum is measured in TVDSS metres; all alignment and correlation remain in TWT milliseconds.


**I-50_COAL exception:** The established I-50_COAL marker is retained as the primary regional anchor regardless of detected coal-bed thickness. The 1.5 m minimum applies only to other automatically detected coal beds. No artificial bed interval is created for I-50_COAL.


Deviation surveys and checkshots are loaded from `/home/apgce_team_08/To_Overlay_Horizon/`. No deviation file implies vertical; checkshot is mandatory. For vertical wells, absolute X/Y requires a wellhead record or `WELLHEAD_XY_OVERRIDES`.

In [12]:
# CELL 12 — Export all screened markers with trajectory-correct X/Y and TWT
# Reuse EXACT trajectory and time models from Cells 3 and 4 (no second conversion).
import numpy as np, pandas as pd
from IPython.display import display
EXPORT_UNAPPROVED_ADJUSTMENTS=False
OUTPUT_CSV=OUTPUT_ROOT/'Phase5_Proposed_Markers_X_Y_TWT_All_7_Wells.csv'
def _xy_at_twt(well,twt):
    df=WELL_DATA[well]
    md=float(_strict_interp([twt],df['TWT_ms'],df['MDKB_m'])[0])
    if not np.isfinite(md):return np.nan,np.nan
    tr=TRAJECTORIES[well]
    if tr['mode']=='VERTICAL':return _wellhead_xy(well)
    if tr['X'] is None or tr['Y'] is None:
        # Deviation survey may have MD/TVD but not absolute coordinates.
        # Do not replace a deviated well with surface X/Y.
        return np.nan,np.nan
    x=float(_strict_interp([md],tr['MD'],tr['X'])[0])
    y=float(_strict_interp([md],tr['MD'],tr['Y'])[0])
    return x,y

# Preserve measured ties. Only use screened, noncrossing missing-marker proposals.
markers=list(MATRIX.Marker_Name)
by_well={}
for well in WELL_NAMES:
    ties=MEASURED[MEASURED.Well==well]
    measured=ties.groupby('Canonical')['TWT_ms'].median().to_dict()
    accepted={}
    subset=PROPOSALS[PROPOSALS.Well==well]
    for _,p in subset.iterrows():
        name=p.Canonical
        if name not in markers:continue
        decision=str(p.QC_Decision)
        if name in measured:
            if EXPORT_UNAPPROVED_ADJUSTMENTS and decision=='ADJUSTMENT_FOR_REVIEW' and np.isfinite(p.Proposed_TWT_ms):
                accepted[name]=float(p.Proposed_TWT_ms)
            else:
                accepted[name]=float(measured[name])
        elif decision=='PROVISIONAL_NONCROSSING' and np.isfinite(p.Proposed_TWT_ms):
            accepted[name]=float(p.Proposed_TWT_ms)
    for name,t in measured.items():
        if name in markers and np.isfinite(t) and name not in accepted:accepted[name]=float(t)
    by_well[well]=accepted

rows=[]; qc=[]
for name in markers:
    row={'Marker_Name':name}
    for well in WELL_NAMES:
        n=well.split('_')[1]
        twt=by_well[well].get(name,np.nan)
        x,y=(np.nan,np.nan)
        if np.isfinite(twt):x,y=_xy_at_twt(well,twt)
        row[f'Well-{n} X']=x;row[f'Well-{n} Y']=y;row[f'Well-{n} TWT']=twt
        if np.isfinite(twt) and (not np.isfinite(x) or not np.isfinite(y)):
            qc.append({'Marker':name,'Well':well,'TWT_ms':twt,
                       'Reason':'No absolute trajectory X/Y or verified wellhead X/Y'})
    rows.append(row)
export=pd.DataFrame(rows)
cols=['Marker_Name']+[f'Well-{w.split("_")[1]} {v}' for w in WELL_NAMES for v in ('X','Y','TWT')]
export=export[cols]
# Export in the interpreter-defined shallow-to-deep order, NOT median TWT.
# Unlisted markers are appended after ranked markers and ordered by median TWT.
# Preserve all original TWT and trajectory-derived X/Y values.
twt_cols=[f'Well-{w.split("_")[1]} TWT' for w in WELL_NAMES]
median_twt=export[twt_cols].apply(pd.to_numeric,errors='coerce').median(axis=1,skipna=True)
rank=export['Marker_Name'].map(STRATIGRAPHIC_RANK)
unlisted=export.loc[rank.isna(),'Marker_Name'].tolist()
if unlisted:
    print('WARNING: markers absent from explicit stratigraphic order (appended last):',unlisted)
export=(export.assign(_rank=rank.fillna(len(STRATIGRAPHIC_ORDER)),
                      _unlisted_median=median_twt)
        .sort_values(['_rank','_unlisted_median'],kind='stable',na_position='last')
        .drop(columns=['_rank','_unlisted_median']).reset_index(drop=True))
print('Marker rows ordered by explicit interpreter stratigraphy (shallowest to deepest).')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
export.to_csv(OUTPUT_CSV,index=False,float_format='%.6f')
# Second CSV: keep markers with valid TWT ties in at least two distinct wells.
# Count TWT (not X/Y) so a missing absolute coordinate does not wrongly
# remove an otherwise correlated marker. Preserve the same shallow-to-deep order.
OUTPUT_CSV_MULTI_WELL=OUTPUT_ROOT/'Phase5_Proposed_Markers_X_Y_TWT_At_Least_2_Wells.csv'
well_twt_count=export[twt_cols].apply(pd.to_numeric,errors='coerce').notna().sum(axis=1)
export_multi_well=export.loc[well_twt_count >= 2].copy()
export_multi_well.to_csv(OUTPUT_CSV_MULTI_WELL,index=False,float_format='%.6f')
print('Filtered CSV:',OUTPUT_CSV_MULTI_WELL,
      '| Markers in >=2 wells:',len(export_multi_well),
      '| Single-well or empty markers omitted:',len(export)-len(export_multi_well))
qc_df=pd.DataFrame(qc,columns=['Marker','Well','TWT_ms','Reason'])
qc_df.to_csv(OUTPUT_ROOT/'Phase5_Marker_XY_Missing_QC.csv',index=False)
print('CSV:',OUTPUT_CSV,'Rows:',len(export),'Columns:',len(export.columns))
print('Marker coordinates without verified absolute X/Y:',len(qc_df))
if len(qc_df):display(qc_df.head(20))
display(export.head(8))
print('Cell 12 completed')

Well_1: LAS wellhead X=470216.400, Y=577913.780 (fields X, Y; verify CRS/units)
Well_7: LAS wellhead X=462597.310, Y=579774.760 (fields X, Y; verify CRS/units)
Well_8: LAS wellhead X=469356.850, Y=570185.410 (fields X, Y; verify CRS/units)
Well_1: LAS wellhead X=470216.400, Y=577913.780 (fields X, Y; verify CRS/units)
Well_7: LAS wellhead X=462597.310, Y=579774.760 (fields X, Y; verify CRS/units)
Well_8: LAS wellhead X=469356.850, Y=570185.410 (fields X, Y; verify CRS/units)
Well_1: LAS wellhead X=470216.400, Y=577913.780 (fields X, Y; verify CRS/units)
Well_7: LAS wellhead X=462597.310, Y=579774.760 (fields X, Y; verify CRS/units)
Well_1: LAS wellhead X=470216.400, Y=577913.780 (fields X, Y; verify CRS/units)
Well_7: LAS wellhead X=462597.310, Y=579774.760 (fields X, Y; verify CRS/units)
Well_1: LAS wellhead X=470216.400, Y=577913.780 (fields X, Y; verify CRS/units)
Well_7: LAS wellhead X=462597.310, Y=579774.760 (fields X, Y; verify CRS/units)
Well_8: LAS wellhead X=469356.850, Y=570

,Marker_Name,Well-1 X,Well-1 Y,Well-1 TWT,Well-2 X,Well-2 Y,Well-2 TWT,Well-4 X,Well-4 Y,Well-4 TWT,...,Well-5 TWT,Well-6 X,Well-6 Y,Well-6 TWT,Well-7 X,Well-7 Y,Well-7 TWT,Well-8 X,Well-8 Y,Well-8 TWT
0,BASAL_B_UNCONFORMITY,NaN,NaN,NaN,471455.819297,573422.947124,947.426400,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TOP_GROUP_H,470216.4,577913.78,1263.815229,471456.686552,573426.082471,1283.920400,470944.886259,575662.051209,1246.595099,...,NaN,NaN,NaN,NaN,462597.31,579774.76,1262.604324,NaN,NaN,NaN
2,TOP_GROUP_I,470216.4,577913.78,1511.826903,471457.971902,573428.984135,1496.671000,470946.317548,575661.896959,1470.491455,...,1496.268667,463912.3,570046.258745,1493.109846,462597.31,579774.76,1488.271228,469356.85,570185.41,1537.54026
3,I-15.0L,470216.4,577913.78,1544.230312,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,462597.31,579774.76,1538.080754,469356.85,570185.41,1581.40528
4,I-25,470216.4,577913.78,1550.744211,471458.435458,573430.086458,1551.876000,470946.375749,575662.056866,1528.389525,...,1559.905116,463912.3,570047.004033,1536.743210,462597.31,579774.76,1541.870496,NaN,NaN,NaN
5,I-27.0,470216.4,577913.78,1561.088153,471458.588103,573430.449443,1569.981318,NaN,NaN,NaN,...,NaN,463912.3,570047.290231,1553.450052,462597.31,579774.76,1552.442517,469356.85,570185.41,1599.09888
6,I-35,470216.4,577913.78,1567.076471,471458.624314,573430.535553,1574.316571,470946.373394,575662.028237,1548.981218,...,1568.472558,463912.3,570047.407222,1559.970272,462597.31,579774.76,1557.544167,469356.85,570185.41,1605.38408
7,I-35_LOWER,470216.4,577913.78,1576.915308,471458.716302,573430.760943,1585.384607,NaN,NaN,NaN,...,1588.977281,463912.3,570047.584698,1569.861480,462597.31,579774.76,1570.129167,469356.85,570185.41,1615.30808


Cell 12 completed


In [15]:
# Cell 13 — Phase 4 water/oil/gas intervals paired with nearest Phase 5 marker (TWT only)
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

ROOT = Path('/home/apgce_team_08/link-to-apgce_team_08')
CLASSES_PATH = ROOT / 'Phase4_7Wells_Reservoir_Classification_TWT.csv'
MARKERS_PATH = ROOT / 'Phase5_Well_Correlation_Output' / 'Phase5_Proposed_Markers_X_Y_TWT_At_Least_2_Wells.csv'
OUT_DIR = ROOT / 'Phase5_Well_Correlation_Output'
MAX_SAMPLE_GAP_MS = 10.0  # TWT discontinuity that splits an interval
FLUID_FILTER = [2,3]  # choose [1] water, [2] oil, [3] gas, [2,3] hydrocarbons
FLUID_NAMES = {1: 'Water', 2: 'Oil', 3: 'Gas'}

if not set(FLUID_FILTER).issubset({1, 2, 3}):
    raise ValueError('FLUID_FILTER must contain only 1 (water), 2 (oil), 3 (gas)')
for path in [CLASSES_PATH, MARKERS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)
c = pd.read_csv(CLASSES_PATH)
m = pd.read_csv(MARKERS_PATH)
needed = {'Well', 'TWT_ms', 'ML_FLUID_FLAG'}
if not needed.issubset(c.columns):
    raise ValueError(f'Phase 4 CSV missing {sorted(needed - set(c.columns))}. Rerun Phase 4 Cell 19 with ML_FLUID_FLAG retained.')
if 'Marker_Name' not in m:
    raise ValueError('Marker CSV must contain Marker_Name')

all_records = []
for num in (1, 2, 4, 5, 6, 7, 8):
    well = f'Well_{num}'
    part = c.loc[c['Well'].astype(str).str.replace('-', '_', regex=False).eq(well)].copy()
    if part.empty:
        raise ValueError(f'No Phase 4 classification samples found for {well}')
    part['TWT_ms'] = pd.to_numeric(part['TWT_ms'], errors='coerce')
    part['ML_FLUID_FLAG'] = pd.to_numeric(part['ML_FLUID_FLAG'], errors='coerce')
    # Preserve source ordering if present; otherwise sort by TWT.
    part = part.sort_values('TWT_ms', kind='stable').reset_index(drop=True)
    twt = part['TWT_ms'].to_numpy(float)
    cls = part['ML_FLUID_FLAG'].to_numpy(float)
    # Break intervals at a fluid-class change, unknown/nonreservoir samples,
    # large TWT gaps, or gaps in original source-row numbering.
    breaks = np.ones(len(part), dtype=bool)
    if len(part) > 1:
        breaks[1:] = (cls[1:] != cls[:-1]) | (~np.isfinite(twt[1:])) | (~np.isfinite(twt[:-1])) | (np.diff(twt) > MAX_SAMPLE_GAP_MS)
        if 'SOURCE_ROW' in part.columns:
            src = pd.to_numeric(part['SOURCE_ROW'], errors='coerce').to_numpy(float)
            breaks[1:] |= (~np.isfinite(src[1:])) | (~np.isfinite(src[:-1])) | (np.abs(np.diff(src)) != 1)
    group_ids = np.cumsum(breaks)

    twtcol, xcol, ycol = f'Well-{num} TWT', f'Well-{num} X', f'Well-{num} Y'
    if twtcol not in m:
        raise ValueError(f'Marker CSV missing {twtcol}')
    candidates = m[['Marker_Name', twtcol] + [col for col in (xcol, ycol) if col in m]].copy()
    candidates[twtcol] = pd.to_numeric(candidates[twtcol], errors='coerce')
    candidates = candidates.dropna(subset=[twtcol])
    valid = np.isfinite(twt) & np.isin(cls, [1, 2, 3])
    for interval_num, (_, indices) in enumerate(part.loc[valid].groupby(group_ids[valid]).groups.items(), 1):
        segment = part.loc[list(indices)].sort_values('TWT_ms')
        fluid = int(segment['ML_FLUID_FLAG'].iloc[0])
        top, base = float(segment['TWT_ms'].iloc[0]), float(segment['TWT_ms'].iloc[-1])
        mid = (top + base) / 2.0
        row = {'Well': well, 'Interval_ID': f'{well}_R{interval_num:04d}',
               'Fluid_Class': fluid, 'Fluid_Type': FLUID_NAMES[fluid],
               'Top_TWT_ms': top, 'Base_TWT_ms': base, 'Mid_TWT_ms': mid,
               'Interval_TWT_ms': base - top, 'Samples': len(segment),
               'Nearest_Marker': pd.NA, 'Marker_TWT_ms': np.nan,
               'Delta_TWT_ms': np.nan, 'Abs_Delta_TWT_ms': np.nan,
               'Marker_X': np.nan, 'Marker_Y': np.nan}
        if not candidates.empty:
            pick = candidates.loc[(candidates[twtcol] - mid).abs().idxmin()]
            delta = mid - float(pick[twtcol])
            row.update(Nearest_Marker=str(pick['Marker_Name']),
                       Marker_TWT_ms=float(pick[twtcol]), Delta_TWT_ms=delta,
                       Abs_Delta_TWT_ms=abs(delta),
                       Marker_X=pd.to_numeric(pick.get(xcol, np.nan), errors='coerce'),
                       Marker_Y=pd.to_numeric(pick.get(ycol, np.nan), errors='coerce'))
        all_records.append(row)

if not all_records:
    raise RuntimeError('No valid water/oil/gas intervals found; check ML_FLUID_FLAG and TWT_ms')
all_df = pd.DataFrame(all_records)
filtered_df = all_df.loc[all_df['Fluid_Class'].isin(FLUID_FILTER)].copy()
OUT_DIR.mkdir(parents=True, exist_ok=True)
all_path = OUT_DIR / 'Phase5_Fluid_Intervals_Nearest_Markers_All.csv'
filtered_path = OUT_DIR / 'Phase5_Fluid_Intervals_Nearest_Markers_Filtered.csv'
all_df.to_csv(all_path, index=False)
filtered_df.to_csv(filtered_path, index=False)
print('Fluid filter:', ', '.join(f'{i}={FLUID_NAMES[i]}' for i in FLUID_FILTER))
print('All-fluid CSV:', all_path)
print('Filtered CSV:', filtered_path)
print('Counts by well and fluid (all classes):')
display(pd.crosstab(all_df['Well'], all_df['Fluid_Type']).reindex(columns=['Water', 'Oil', 'Gas'], fill_value=0))
print('Cell 13 completed')


Fluid filter: 2=Oil, 3=Gas
All-fluid CSV: /home/apgce_team_08/link-to-apgce_team_08/Phase5_Well_Correlation_Output/Phase5_Fluid_Intervals_Nearest_Markers_All.csv
Filtered CSV: /home/apgce_team_08/link-to-apgce_team_08/Phase5_Well_Correlation_Output/Phase5_Fluid_Intervals_Nearest_Markers_Filtered.csv
Counts by well and fluid (all classes):


Fluid_Type,Water,Oil,Gas
Well,,,
Well_1,183,0,0
Well_2,128,0,0
Well_4,122,5,1
Well_5,136,3,0
Well_6,98,0,0
Well_7,135,3,0
Well_8,99,1,0


Cell 13 completed
